# Agentic semantic policy--sentiment gap analysis

This notebook performs an LLM-assisted, evidence-grounded semantic-gap analysis over the shared cleaned sentence inventory. Frozen NMF topic assignments are loaded directly from the preceding topic-modelling stage only to stratify final diagnostics by native topic ID. Synthetic sentiment is used only as a controlled robustness perturbation of the agentic workflow, globally and for each eligible country scope; synthetic outputs never become substantive findings.


In [1]:
from __future__ import annotations

import hashlib
import itertools
import json
import math
import os
import random
import re
import sys
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any, Iterable

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 240)

PIPELINE_VERSION = "agentic-semantic-v3.1"
PROMPT_VERSION = "direct-comparability-v3.1"

# Reproducibility and workload
RANDOM_STATE = int(os.environ.get("AGENT_RANDOM_STATE", "42"))
TARGET_USABLE_RUNS = max(2, int(os.environ.get("AGENT_TARGET_USABLE_RUNS", "3")))
MAX_RUN_ATTEMPTS = max(TARGET_USABLE_RUNS, int(os.environ.get("AGENT_MAX_RUN_ATTEMPTS", "5")))
BATCHES_PER_SCOPE = max(1, int(os.environ.get("AGENT_BATCHES_PER_SCOPE", "3")))
SENTENCES_PER_CORPUS = max(8, int(os.environ.get("AGENT_SENTENCES_PER_CORPUS", "24")))
CAUSAL_HINT_SHARE = min(1.0, max(0.0, float(os.environ.get("AGENT_CAUSAL_HINT_SHARE", "0.85"))))
MAX_FINDINGS = max(1, int(os.environ.get("AGENT_MAX_FINDINGS", "5")))
MAX_CASES = max(0, int(os.environ.get("AGENT_MAX_CASES", "0")))

# Scope eligibility and retention criteria
MIN_SCOPE_SENTENCES = max(1, int(os.environ.get("AGENT_MIN_SCOPE_SENTENCES", "12")))
MIN_SCOPE_SOURCES = max(1, int(os.environ.get("AGENT_MIN_SCOPE_SOURCES", "2")))
MIN_CONFIDENCE = min(1.0, max(0.0, float(os.environ.get("AGENT_MIN_CONFIDENCE", "0.60"))))
MIN_FAITHFULNESS = min(1.0, max(0.0, float(os.environ.get("AGENT_MIN_FAITHFULNESS", "0.80"))))
MIN_MATCH_SCORE = min(1.0, max(0.0, float(os.environ.get("AGENT_MIN_MATCH_SCORE", "0.50"))))
MIN_RECURRENT_RUNS = max(2, int(os.environ.get("AGENT_MIN_RECURRENT_RUNS", "2")))
MIN_RECURRENT_RUNS = min(MIN_RECURRENT_RUNS, TARGET_USABLE_RUNS)
MIN_CLASSIFICATION_AGREEMENT = min(1.0, max(0.0, float(os.environ.get("AGENT_MIN_CLASSIFICATION_AGREEMENT", "0.66"))))
MIN_RELATION_AGREEMENT = min(1.0, max(0.0, float(os.environ.get("AGENT_MIN_RELATION_AGREEMENT", "0.50"))))
MIN_DIRECT_COMPARABILITY_SHARE = min(1.0, max(0.0, float(os.environ.get("AGENT_MIN_DIRECT_COMPARABILITY_SHARE", "0.66"))))
REQUIRE_DIRECT_COMPARABILITY = os.environ.get("AGENT_REQUIRE_DIRECT_COMPARABILITY", "1").strip().lower() not in {"0", "false", "no", "off"}
REQUIRE_FROZEN_TOPICS = os.environ.get("AGENT_REQUIRE_FROZEN_TOPICS", "1").strip().lower() not in {"0", "false", "no", "off"}

RUN_AGENT = os.environ.get(
    "RUN_DEEPSEEK_AGENT",
    "1" if os.environ.get("DEEPSEEK_API_KEY") else "0",
).strip().lower() not in {"0", "false", "no", "off"}
RESUME_RUNS = os.environ.get("AGENT_RESUME_RUNS", "1").strip().lower() not in {
    "0", "false", "no", "off"
}
RUN_SYNTHETIC_ROBUSTNESS = os.environ.get("AGENT_SYNTHETIC_ROBUSTNESS", "1").strip().lower() not in {
    "0", "false", "no", "off"
}


def find_method_root() -> Path:
    configured = os.environ.get("CAUSAL_NLP_ROOT")
    candidates: list[Path] = []
    if configured:
        candidates.append(Path(configured).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents, cwd / "progress" / "causal_nlp"])

    checked: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in checked:
            continue
        checked.add(candidate)
        if (
            (candidate / "agent.py").exists()
            and (candidate / "output" / "shared" / "clean_sentence_inventory.csv").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate agent.py and output/shared/clean_sentence_inventory.csv. "
        "Set CAUSAL_NLP_ROOT to the causal_nlp directory after running causal_text_cleaning.py."
    )


METHOD_ROOT = find_method_root()

def find_project_root(method_root: Path) -> Path:
    configured = os.environ.get("CAUSAL_NLP_PROJECT_ROOT")
    candidates = []
    if configured:
        candidates.append(Path(configured).expanduser().resolve())
    candidates.extend([method_root, *method_root.parents])
    for candidate in candidates:
        nmf = candidate / "progress" / "topic_modelling" / "nmf"
        if nmf.exists():
            return candidate.resolve()
    if REQUIRE_FROZEN_TOPICS:
        raise FileNotFoundError(
            "Frozen NMF outputs were not found. Set CAUSAL_NLP_PROJECT_ROOT to the msc project root. "
            "The agentic text analysis still uses only the shared clean sentence inventory; NMF files are "
            "used only for native-topic stratification of the final diagnostics."
        )
    return method_root.resolve()

PROJECT_ROOT = find_project_root(METHOD_ROOT)
NMF_ROOT = PROJECT_ROOT / "progress" / "topic_modelling" / "nmf"
POLICY_NMF_ASSIGNMENTS = NMF_ROOT / "policy" / "output" / "global" / "policy_global_nmf_documents_with_topic_labels_final.csv"
SENTIMENT_NMF_ASSIGNMENTS = NMF_ROOT / "sentiment" / "output" / "sentiment_nmf_original_documents_with_topic_labels_final.csv"
SYNTHETIC_NMF_ASSIGNMENTS = NMF_ROOT / "sentiment" / "output" / "sentiment_nmf_synthetic_assignments_with_topic_labels_final.csv"

SHARED_DIR = METHOD_ROOT / "output" / "shared"
SHARED_INPUT = SHARED_DIR / "clean_sentence_inventory.csv"
SHARED_METADATA = SHARED_DIR / "clean_sentence_inventory_metadata.json"
SHARED_VALIDATION = SHARED_DIR / "clean_sentence_inventory_validation.csv"

OUTPUT_DIR = METHOD_ROOT / "output" / "agentic_semantic_gap"
IMG_DIR = METHOD_ROOT / "img" / "agentic_semantic_gap"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMG_DIR.mkdir(parents=True, exist_ok=True)

if str(METHOD_ROOT) not in sys.path:
    sys.path.insert(0, str(METHOD_ROOT))
from agent import ask_agent

# Main empirical outputs
RUNS_PATH = OUTPUT_DIR / "semantic_agent_runs.jsonl"
EVIDENCE_PATH = OUTPUT_DIR / "semantic_agent_evidence.csv"
STABILITY_PATH = OUTPUT_DIR / "semantic_agent_stability.csv"
CANDIDATES_PATH = OUTPUT_DIR / "semantic_agent_candidates.csv"
FINDINGS_PATH = OUTPUT_DIR / "semantic_agent_findings.csv"
FINDING_EVIDENCE_PATH = OUTPUT_DIR / "semantic_agent_finding_evidence.csv"
REVIEW_PATH = OUTPUT_DIR / "semantic_agent_human_review.csv"
EVALUATION_PATH = OUTPUT_DIR / "semantic_agent_evaluation.csv"
DIRECTION_SUMMARY_PATH = OUTPUT_DIR / "semantic_agent_direction_summary.csv"
RUN_ISSUES_PATH = OUTPUT_DIR / "semantic_agent_run_issues.csv"
NATIVE_TOPIC_DEFICIT_PATH = OUTPUT_DIR / "semantic_agent_deficit_by_native_topic.csv"
NATIVE_TOPIC_CONNECTION_PATH = OUTPUT_DIR / "semantic_agent_native_topic_connections.csv"
COUNTRY_NATIVE_TOPIC_DEFICIT_PATH = OUTPUT_DIR / "semantic_agent_country_deficit_by_native_topic.csv"
DEFICIT_DISTRIBUTION_PATH = OUTPUT_DIR / "semantic_agent_directional_deficits.csv"

# Synthetic robustness outputs. These never feed the main findings.
ROBUSTNESS_EVIDENCE_PATH = OUTPUT_DIR / "semantic_agent_robustness_evidence.csv"
ROBUSTNESS_RUNS_PATH = OUTPUT_DIR / "semantic_agent_robustness_runs.jsonl"
ROBUSTNESS_SUMMARY_PATH = OUTPUT_DIR / "semantic_agent_robustness_summary.csv"
ROBUSTNESS_SCOPE_SUMMARY_PATH = OUTPUT_DIR / "semantic_agent_robustness_by_scope.csv"

RUN_SUMMARY_PATH = OUTPUT_DIR / "semantic_agent_run_summary.csv"
OUTPUT_STATUS_PATH = OUTPUT_DIR / "semantic_agent_output_status.csv"
PNG_VALIDATION_PATH = OUTPUT_DIR / "semantic_agent_png_validation.csv"

# Removed from v2.1: synthetic probe findings are no longer exported as substantive-style candidates.
DEPRECATED_OUTPUTS = [
    OUTPUT_DIR / "semantic_agent_robustness_candidates.csv",
]


def remove_stale(path: Path) -> None:
    if path.exists():
        path.unlink()


def write_nonempty_csv(frame: pd.DataFrame, path: Path) -> bool:
    if frame is not None and not frame.empty:
        frame.to_csv(path, index=False)
        return True
    remove_stale(path)
    return False


for deprecated_path in DEPRECATED_OUTPUTS:
    remove_stale(deprecated_path)


print("Method folder:", METHOD_ROOT)
print("Shared input:", SHARED_INPUT)
print("Output folder:", OUTPUT_DIR)
print("Pipeline version:", PIPELINE_VERSION)
print("DeepSeek enabled:", RUN_AGENT)
print("Synthetic robustness enabled:", RUN_SYNTHETIC_ROBUSTNESS)
print("Target usable runs per package:", TARGET_USABLE_RUNS)
print("Maximum run attempts per package:", MAX_RUN_ATTEMPTS)
print("Required recurrent runs:", MIN_RECURRENT_RUNS)
print("Minimum cross-run match score:", MIN_MATCH_SCORE)
print("Minimum classification agreement:", MIN_CLASSIFICATION_AGREEMENT)
print("Frozen topic stratification:", REQUIRE_FROZEN_TOPICS)


Method folder: /home/nsirim/Github/mscdsa/msc/progress/causal_nlp
Shared input: /home/nsirim/Github/mscdsa/msc/progress/causal_nlp/output/shared/clean_sentence_inventory.csv
Output folder: /home/nsirim/Github/mscdsa/msc/progress/causal_nlp/output/agentic_semantic_gap
Pipeline version: agentic-semantic-v3.1
DeepSeek enabled: True
Synthetic robustness enabled: True
Target usable runs per package: 3
Maximum run attempts per package: 5
Required recurrent runs: 2
Minimum cross-run match score: 0.5
Minimum classification agreement: 0.66
Frozen topic stratification: True


In [2]:
REQUIRED_COLUMNS = {
    "sentence_id", "clean_sentence", "corpus", "source_type", "source_file",
    "doc_id", "chunk_id", "country", "heading_context", "synthetic_type",
    "cleaning_version",
}

inventory = pd.read_csv(SHARED_INPUT)
missing = REQUIRED_COLUMNS.difference(inventory.columns)
if missing:
    raise ValueError(f"Shared inventory is missing columns: {sorted(missing)}")

for column in REQUIRED_COLUMNS:
    inventory[column] = inventory[column].fillna("").astype(str)

if inventory["sentence_id"].duplicated().any():
    examples = inventory.loc[inventory["sentence_id"].duplicated(keep=False), "sentence_id"].head(10).tolist()
    raise ValueError(f"sentence_id must be unique. Examples: {examples}")
if inventory["cleaning_version"].nunique() != 1:
    raise ValueError("The shared inventory contains more than one cleaning version.")

metadata: dict[str, Any] = {}
if SHARED_METADATA.exists():
    metadata = json.loads(SHARED_METADATA.read_text(encoding="utf-8"))
    expected_rows = metadata.get("inventory_rows")
    if expected_rows is not None and int(expected_rows) != len(inventory):
        raise ValueError("Shared inventory row count does not match its metadata file.")

if SHARED_VALIDATION.exists():
    shared_validation = pd.read_csv(SHARED_VALIDATION).fillna("")
    if "status" in shared_validation.columns and not shared_validation["status"].eq("passed").all():
        raise ValueError("Shared cleaning validation contains a failed check.")

synthetic_mask = (
    inventory["source_type"].str.strip().str.lower().eq("synthetic")
    | inventory["synthetic_type"].str.strip().ne("")
)

empirical = inventory[
    ~synthetic_mask
    & inventory["corpus"].str.strip().str.lower().isin(["policy", "sentiment"])
].copy()
synthetic = inventory[
    synthetic_mask
    & inventory["corpus"].str.strip().str.lower().eq("sentiment")
].copy()

for frame in [empirical, synthetic]:
    frame["corpus"] = frame["corpus"].str.strip().str.lower()
    frame["clean_sentence"] = frame["clean_sentence"].str.replace(r"\s+", " ", regex=True).str.strip()

empirical = empirical[empirical["clean_sentence"].ne("")].copy()
synthetic = synthetic[synthetic["clean_sentence"].ne("")].copy()


def infer_country(row: pd.Series) -> str:
    current = str(row.get("country", "")).strip().lower()
    if current and current not in {"nan", "none", "other", "unknown", "synthetic_sentiment"}:
        return re.sub(r"\s+", "_", current)

    source = " ".join(
        str(row.get(column, ""))
        for column in ["source_file", "doc_id", "heading_context"]
    ).lower()
    country_tokens = {
        "ireland": ["ireland", "irish", "qqi"],
        "france": ["france", "french", "français", "francais", "ifop"],
        "australia": ["australia", "australian"],
        "united_states": ["united states", "usa", "u.s.", "american"],
    }
    for country, tokens in country_tokens.items():
        if any(token in source for token in tokens):
            return country
    return "other"


empirical["analysis_country"] = empirical.apply(infer_country, axis=1)
synthetic["analysis_country"] = "synthetic_probe"


def _load_topic_assignments(path: Path, topic_column: str, prefix: str) -> pd.DataFrame:
    if not path.exists():
        if REQUIRE_FROZEN_TOPICS:
            raise FileNotFoundError(f"Required frozen NMF assignment file not found: {path}")
        return pd.DataFrame(columns=["chunk_id", "native_topic", "topic_code", "topic_space"])
    frame = pd.read_csv(path)
    required = {"chunk_id", topic_column}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"Frozen topic assignment file {path.name} is missing {sorted(missing)}")
    frame = frame[["chunk_id", topic_column]].drop_duplicates("chunk_id").copy()
    frame["native_topic"] = pd.to_numeric(frame[topic_column], errors="coerce")
    frame = frame.dropna(subset=["native_topic"])
    frame["native_topic"] = frame["native_topic"].astype(int)
    frame["topic_code"] = frame["native_topic"].map(lambda value: f"{prefix}{int(value)}")
    frame["topic_space"] = "policy" if prefix == "P" else "sentiment"
    return frame[["chunk_id", "native_topic", "topic_code", "topic_space"]]


policy_topic_map = _load_topic_assignments(POLICY_NMF_ASSIGNMENTS, "topic", "P")
sentiment_topic_map = _load_topic_assignments(SENTIMENT_NMF_ASSIGNMENTS, "topic", "S")
synthetic_topic_map = _load_topic_assignments(SYNTHETIC_NMF_ASSIGNMENTS, "assigned_topic", "S")

empirical_policy = empirical[empirical["corpus"].eq("policy")].merge(
    policy_topic_map, on="chunk_id", how="left", validate="many_to_one"
)
empirical_sentiment = empirical[empirical["corpus"].eq("sentiment")].merge(
    sentiment_topic_map, on="chunk_id", how="left", validate="many_to_one"
)
empirical = pd.concat([empirical_policy, empirical_sentiment], ignore_index=True)
synthetic = synthetic.merge(
    synthetic_topic_map, on="chunk_id", how="left", validate="many_to_one"
)

for frame_name, frame in [("empirical", empirical), ("synthetic", synthetic)]:
    linked = frame["native_topic"].notna().mean() if len(frame) else 1.0
    print(f"Frozen-topic linkage ({frame_name}): {linked:.3f}")
    if REQUIRE_FROZEN_TOPICS and linked < 0.95:
        raise ValueError(f"Frozen-topic linkage for {frame_name} fell below 95%: {linked:.3f}")

# Broad multilingual causal-language flag. It is deliberately permissive because
# it is used only to prioritise evidence for the LLM, never to define a finding.
CAUSAL_HINT_RE = re.compile(
    r"\b(?:"
    r"because(?:\s+of)?|due\s+to|owing\s+to|as\s+a\s+result(?:\s+of)?|"
    r"result(?:s|ed|ing)?\s+(?:in|from)|lead(?:s|ing)?\s+to|gave?\s+rise\s+to|"
    r"cause(?:s|d|ing)?|contribut(?:e|es|ed|ing)\s+to|driv(?:e|es|en|ing)|"
    r"produc(?:e|es|ed|ing)|trigger(?:s|ed|ing)?|generat(?:e|es|ed|ing)|"
    r"creat(?:e|es|ed|ing)|exacerbat(?:e|es|ed|ing)|worsen(?:s|ed|ing)?|"
    r"intensif(?:y|ies|ied|ying)|accelerat(?:e|es|ed|ing)|increas(?:e|es|ed|ing)|"
    r"rais(?:e|es|ed|ing)|amplif(?:y|ies|ied|ying)|heighten(?:s|ed|ing)?|"
    r"reduc(?:e|es|ed|ing)|prevent(?:s|ed|ing)?|limit(?:s|ed|ing)?|"
    r"mitigat(?:e|es|ed|ing)|decreas(?:e|es|ed|ing)|lower(?:s|ed|ing)|"
    r"minimi(?:s|z)(?:e|es|ed|ing)|alleviat(?:e|es|ed|ing)|avoid(?:s|ed|ing)?|"
    r"curb(?:s|ed|ing)?|constrain(?:s|ed|ing)?|protect(?:s|ed|ing)?|"
    r"safeguard(?:s|ed|ing)?|enabl(?:e|es|ed|ing)|support(?:s|ed|ing)?|"
    r"facilitat(?:e|es|ed|ing)|allow(?:s|ed|ing)?|help(?:s|ed|ing)?|"
    r"foster(?:s|ed|ing)?|encourag(?:e|es|ed|ing)|promot(?:e|es|ed|ing)|"
    r"empower(?:s|ed|ing)?|assist(?:s|ed|ing)?|requir(?:e|es|ed|ing)|"
    r"depend(?:s|ed|ing)?\s+on|rel(?:y|ies|ied|ying)(?:\s+on)?|need(?:s|ed|ing)?|"
    r"necessitat(?:e|es|ed|ing)|presuppos(?:e|es|ed|ing)|call(?:s|ed|ing)?\s+for|"
    r"contingent\s+on|conditional\s+on|risk(?:s|ed|ing)?|threaten(?:s|ed|ing)?|"
    r"undermin(?:e|es|ed|ing)|harm(?:s|ed|ing)?|jeopardi(?:s|z)(?:e|es|ed|ing)|"
    r"endanger(?:s|ed|ing)?|expos(?:e|es|ed|ing)|compromis(?:e|es|ed|ing)|"
    r"weaken(?:s|ed|ing)?|improv(?:e|es|ed|ing)|enhanc(?:e|es|ed|ing)|"
    r"strengthen(?:s|ed|ing)?|advanc(?:e|es|ed|ing)|boost(?:s|ed|ing)?|"
    r"optimi(?:s|z)(?:e|es|ed|ing)|ensure(?:s|d|ing)?|designed\s+to|"
    r"intended\s+to|in\s+order\s+to|so\s+as\s+to|with\s+the\s+aim\s+of|"
    r"for\s+the\s+purpose\s+of|with\s+a\s+view\s+to|parce\s+que|en\s+raison\s+de|"
    r"à\s+cause\s+de|du\s+fait\s+de|afin\s+de|dans\s+le\s+but\s+de|en\s+vue\s+de|"
    r"condui(?:t|sent|re)|entraîn\w*|caus\w*|provoqu\w*|engendr\w*|contribu\w*|"
    r"génèr\w*|déclench\w*|augment\w*|rédui\w*|prévien\w*|limit\w*|atténu\w*|"
    r"diminu\w*|évit\w*|permet\w*|soutien\w*|facilit\w*|favoris\w*|encourag\w*|"
    r"aid\w*|nécessit\w*|dépend\w*|repos\w*|exig\w*|requi\w*|suppos\w*|appel\w*|"
    r"risqu\w*|menac\w*|compromet\w*|nui\w*|amélior\w*|renforc\w*|optimis\w*"
    r")\b",
    flags=re.IGNORECASE,
)

for frame in [empirical, synthetic]:
    frame["causal_hint"] = frame["clean_sentence"].map(lambda text: bool(CAUSAL_HINT_RE.search(text)))

if empirical.empty:
    raise ValueError("No empirical policy or sentiment sentences were found.")

support = empirical.groupby("corpus", as_index=False).agg(
    sentences=("sentence_id", "count"),
    sources=("source_file", "nunique"),
    causal_hint_sentences=("causal_hint", "sum"),
)
support["causal_hint_rate"] = support["causal_hint_sentences"] / support["sentences"]

synthetic_support = pd.DataFrame([{
    "sentences": len(synthetic),
    "sources": synthetic["source_file"].nunique() if not synthetic.empty else 0,
    "causal_hint_sentences": int(synthetic["causal_hint"].sum()) if not synthetic.empty else 0,
    "causal_hint_rate": float(synthetic["causal_hint"].mean()) if not synthetic.empty else 0.0,
}])

display(support)
display(synthetic_support)

Frozen-topic linkage (empirical): 0.952
Frozen-topic linkage (synthetic): 0.959


,corpus,sentences,sources,causal_hint_sentences,causal_hint_rate
0,policy,15281,56,6604,0.432171
1,sentiment,4182,17,1412,0.337637


,sentences,sources,causal_hint_sentences,causal_hint_rate
0,2128,53,1041,0.489192


In [3]:
ALLOWED_CLASSIFICATIONS = {
    "policy_gap", "sentiment_gap", "partial_alignment", "alignment", "insufficient_evidence",
}
SUBSTANTIVE_CLASSIFICATIONS = {"policy_gap", "sentiment_gap", "partial_alignment"}
ALLOWED_RELATIONS = {
    "causes_or_increases", "reduces_or_prevents", "enables_or_supports",
    "requires_or_depends_on", "risks_or_threatens", "expected_to_improve", "other",
}
ALLOWED_EVIDENCE_QUALITY = {"strong", "adequate", "weak"}
REPORTABLE_CLASSIFICATIONS = {"policy_gap", "sentiment_gap", "partial_alignment", "alignment"}


def evidence_item(row: pd.Series, corpus_override: str | None = None) -> dict[str, Any]:
    return {
        "evidence_id": str(row["sentence_id"]),
        "corpus": corpus_override or str(row["corpus"]),
        "text": str(row["clean_sentence"]),
        "source_file": str(row["source_file"]),
        "doc_id": str(row["doc_id"]),
        "chunk_id": str(row["chunk_id"]),
        "country": str(row["analysis_country"]),
        "heading_context": str(row["heading_context"]),
        "native_topic": int(row["native_topic"]) if pd.notna(row.get("native_topic")) else None,
        "topic_code": str(row.get("topic_code", "")),
        "topic_space": str(row.get("topic_space", "")),
        "causal_hint": bool(row["causal_hint"]),
    }


def round_robin_rows(frame: pd.DataFrame, limit: int, seed: int, corpus_override: str | None = None) -> list[dict[str, Any]]:
    if frame.empty or limit <= 0:
        return []
    rng = random.Random(seed)
    grouped: dict[str, list[dict[str, Any]]] = {}
    for source, group in frame.groupby("source_file", sort=True):
        rows = group.sort_values("sentence_id").to_dict("records")
        rng.shuffle(rows)
        grouped[str(source)] = rows

    sources = list(grouped)
    rng.shuffle(sources)
    selected: list[dict[str, Any]] = []
    while sources and len(selected) < limit:
        remaining: list[str] = []
        for source in sources:
            rows = grouped[source]
            if rows and len(selected) < limit:
                selected.append(evidence_item(pd.Series(rows.pop()), corpus_override=corpus_override))
            if rows:
                remaining.append(source)
        sources = remaining
    return selected


def source_balanced_sample(
    frame: pd.DataFrame,
    limit: int,
    seed: int,
    corpus_override: str | None = None,
    hint_target_override: int | None = None,
) -> list[dict[str, Any]]:
    """Prioritise broad causal-language candidates while preserving source diversity."""
    if frame.empty or limit <= 0:
        return []

    default_hint_target = int(round(limit * CAUSAL_HINT_SHARE))
    hint_target = min(limit, default_hint_target if hint_target_override is None else max(0, hint_target_override))
    hinted = round_robin_rows(frame[frame["causal_hint"]], hint_target, seed, corpus_override)
    chosen_ids = {item["evidence_id"] for item in hinted}
    remaining = frame[~frame["sentence_id"].isin(chosen_ids)]
    if hint_target_override is None:
        rest = round_robin_rows(remaining, limit - len(hinted), seed + 17, corpus_override)
    else:
        # For robustness probes, preserve the baseline causal-hint count when possible.
        nonhint = remaining[~remaining["causal_hint"]]
        rest = round_robin_rows(nonhint, limit - len(hinted), seed + 17, corpus_override)
        if len(rest) < limit - len(hinted):
            rest_ids = {item["evidence_id"] for item in rest}
            fallback = remaining[~remaining["sentence_id"].isin(rest_ids)]
            rest += round_robin_rows(
                fallback, limit - len(hinted) - len(rest), seed + 23, corpus_override
            )
    selected = hinted + rest
    random.Random(seed + 31).shuffle(selected)
    return selected


def build_scopes(frame: pd.DataFrame) -> list[tuple[str, pd.DataFrame]]:
    scopes: list[tuple[str, pd.DataFrame]] = [("global", frame)]
    for country in sorted(frame["analysis_country"].unique()):
        if country == "other":
            continue
        subset = frame[frame["analysis_country"].eq(country)].copy()
        counts = subset.groupby("corpus")["sentence_id"].count().to_dict()
        sources = subset.groupby("corpus")["source_file"].nunique().to_dict()
        if all(counts.get(corpus, 0) >= MIN_SCOPE_SENTENCES for corpus in ["policy", "sentiment"]) and all(
            sources.get(corpus, 0) >= MIN_SCOPE_SOURCES for corpus in ["policy", "sentiment"]
        ):
            scopes.append((country, subset))
    return scopes


def build_empirical_packages(frame: pd.DataFrame) -> tuple[list[dict[str, Any]], pd.DataFrame]:
    packages: list[dict[str, Any]] = []
    evidence_rows: list[dict[str, Any]] = []

    for scope_index, (scope, subset) in enumerate(build_scopes(frame)):
        for batch_index in range(BATCHES_PER_SCOPE):
            seed = RANDOM_STATE + scope_index * 1000 + batch_index * 100
            policy = source_balanced_sample(subset[subset["corpus"].eq("policy")], SENTENCES_PER_CORPUS, seed)
            sentiment = source_balanced_sample(subset[subset["corpus"].eq("sentiment")], SENTENCES_PER_CORPUS, seed + 1)
            if not policy or not sentiment:
                continue

            evidence = policy + sentiment
            random.Random(seed + 2).shuffle(evidence)
            analysis_id = f"{scope}__batch_{batch_index + 1:02d}"
            package = {
                "analysis_id": analysis_id,
                "scope": scope,
                "batch_id": batch_index + 1,
                "maximum_findings": MAX_FINDINGS,
                "evidence_counts": {
                    "policy": len(policy), "sentiment": len(sentiment),
                    "policy_sources": len({item["source_file"] for item in policy}),
                    "sentiment_sources": len({item["source_file"] for item in sentiment}),
                    "policy_causal_hints": sum(bool(item["causal_hint"]) for item in policy),
                    "sentiment_causal_hints": sum(bool(item["causal_hint"]) for item in sentiment),
                },
                "evidence": evidence,
            }
            packages.append(package)
            for item in evidence:
                evidence_rows.append({"analysis_id": analysis_id, "scope": scope, "batch_id": batch_index + 1, **item})

    if MAX_CASES > 0:
        packages = packages[:MAX_CASES]
        kept = {p["analysis_id"] for p in packages}
        evidence_rows = [row for row in evidence_rows if row["analysis_id"] in kept]

    evidence_df = pd.DataFrame(evidence_rows)
    evidence_df.to_csv(EVIDENCE_PATH, index=False)
    return packages, evidence_df


def build_robustness_packages(
    empirical_packages: list[dict[str, Any]],
    synthetic_frame: pd.DataFrame,
) -> tuple[list[dict[str, Any]], pd.DataFrame]:
    """Replace only the sentiment side of every eligible empirical batch with synthetic sentiment.

    The policy evidence is held fixed. Global, France, Ireland, and any other eligible country
    scope are perturbed independently. Synthetic provenance is hidden from the LLM, and probe
    outputs are isolated from substantive findings.
    """
    if not RUN_SYNTHETIC_ROBUSTNESS or synthetic_frame.empty:
        remove_stale(ROBUSTNESS_EVIDENCE_PATH)
        return [], pd.DataFrame()

    packages: list[dict[str, Any]] = []
    rows: list[dict[str, Any]] = []
    probe_bases = list(empirical_packages)

    for index, base in enumerate(probe_bases):
        policy = [dict(item) for item in base["evidence"] if item["corpus"] == "policy"]
        original_sentiment = [dict(item) for item in base["evidence"] if item["corpus"] == "sentiment"]
        original_sentiment_count = len(original_sentiment)
        target_sentiment_sources = len({item["source_file"] for item in original_sentiment})
        target_sentiment_hints = sum(bool(item["causal_hint"]) for item in original_sentiment)

        # Keep batch size, source count, causal-hint count, and policy evidence fixed as closely
        # as possible. Only the sentiment text is perturbed by the synthetic corpus.
        synthetic_sources = sorted(synthetic_frame["source_file"].unique())
        rng = random.Random(RANDOM_STATE + 49000 + index)
        rng.shuffle(synthetic_sources)
        chosen_sources = synthetic_sources[: min(target_sentiment_sources, len(synthetic_sources))]
        synthetic_pool = synthetic_frame[synthetic_frame["source_file"].isin(chosen_sources)].copy()
        synthetic_sentiment = source_balanced_sample(
            synthetic_pool,
            original_sentiment_count,
            RANDOM_STATE + 50000 + index * 100,
            corpus_override="sentiment",
            hint_target_override=target_sentiment_hints,
        )
        if not policy or len(synthetic_sentiment) != original_sentiment_count:
            continue

        evidence = policy + synthetic_sentiment
        random.Random(RANDOM_STATE + 51000 + index).shuffle(evidence)
        analysis_id = f"{base['analysis_id']}__synthetic_probe"
        package = {
            "analysis_id": analysis_id,
            "scope": base["scope"],
            "batch_id": base["batch_id"],
            "probe_of_analysis_id": base["analysis_id"],
            "maximum_findings": MAX_FINDINGS,
            "evidence_counts": {
                "policy": len(policy), "sentiment": len(synthetic_sentiment),
                "policy_sources": len({item["source_file"] for item in policy}),
                "sentiment_sources": len({item["source_file"] for item in synthetic_sentiment}),
                "policy_causal_hints": sum(bool(item["causal_hint"]) for item in policy),
                "sentiment_causal_hints": sum(bool(item["causal_hint"]) for item in synthetic_sentiment),
            },
            "evidence": evidence,
        }
        packages.append(package)
        for item in evidence:
            rows.append({
                "analysis_id": analysis_id,
                "probe_of_analysis_id": base["analysis_id"],
                "scope": base["scope"],
                "batch_id": base["batch_id"],
                **item,
            })

    evidence_df = pd.DataFrame(rows)
    write_nonempty_csv(evidence_df, ROBUSTNESS_EVIDENCE_PATH)
    return packages, evidence_df


def shuffled_package(package: dict[str, Any], seed: int) -> dict[str, Any]:
    value = json.loads(json.dumps(package, ensure_ascii=False))
    is_robustness_probe = "probe_of_analysis_id" in value
    if is_robustness_probe:
        # Blind the LLM to synthetic provenance while retaining immutable evidence IDs.
        sentiment_items = [item for item in value["evidence"] if item["corpus"] == "sentiment"]
        source_names = sorted({item["source_file"] for item in sentiment_items})
        doc_names = sorted({item["doc_id"] for item in sentiment_items})
        source_map = {name: f"sentiment_source_{index+1:03d}" for index, name in enumerate(source_names)}
        doc_map = {name: f"sentiment_document_{index+1:03d}" for index, name in enumerate(doc_names)}
        for item in sentiment_items:
            item["source_file"] = source_map[item["source_file"]]
            item["doc_id"] = doc_map[item["doc_id"]]
            item["country"] = "other"
    random.Random(seed).shuffle(value["evidence"])
    # Do not expose robustness bookkeeping to the LLM.
    value.pop("probe_of_analysis_id", None)
    return value


def valid_ids(package: dict[str, Any], corpus: str | None = None) -> set[str]:
    return {
        item["evidence_id"] for item in package["evidence"]
        if corpus is None or item["corpus"] == corpus
    }


def pairwise_jaccard(values: list[set[str]]) -> float:
    if len(values) < 2:
        return 0.0
    scores=[]
    for left, right in itertools.combinations(values, 2):
        union = left | right
        scores.append(1.0 if not union else len(left & right) / len(union))
    return float(np.mean(scores)) if scores else 0.0


def parse_bool(value: Any) -> bool | None:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    normalised = str(value).strip().lower()
    if normalised in {"1", "true", "yes", "y"}: return True
    if normalised in {"0", "false", "no", "n"}: return False
    return None


def normalise_ids(value: Any) -> list[str]:
    if not isinstance(value, list):
        return []
    return list(dict.fromkeys(str(item).strip() for item in value if str(item).strip()))


def default_directional_deficits(classification: str) -> tuple[float, float]:
    # Never manufacture a numerical deficit from a categorical label. Missing
    # directional scores remain missing and are rejected for reportable findings.
    return np.nan, np.nan


def normalise_findings(value: Any) -> list[dict[str, Any]]:
    if not isinstance(value, list):
        return []
    findings=[]
    for position, item in enumerate(value, start=1):
        if not isinstance(item, dict):
            continue
        record=dict(item)
        record["finding_id"] = str(record.get("finding_id") or f"S{position}").strip()
        record["classification"] = str(record.get("classification", "insufficient_evidence")).strip().lower()
        if record["classification"] not in ALLOWED_CLASSIFICATIONS:
            record["classification"] = "insufficient_evidence"
        record["relation_family"] = str(record.get("relation_family", "other")).strip().lower()
        if record["relation_family"] not in ALLOWED_RELATIONS:
            record["relation_family"] = "other"
        for key in ["cause_theme", "effect_theme", "gap_label", "explanation"]:
            record[key] = re.sub(r"\s+", " ", str(record.get(key, ""))).strip()
        quality = str(record.get("evidence_quality", "adequate")).strip().lower()
        record["evidence_quality"] = quality if quality in ALLOWED_EVIDENCE_QUALITY else "weak"
        for key in ["policy_evidence_ids", "sentiment_evidence_ids", "counterevidence_ids"]:
            record[key] = normalise_ids(record.get(key, []))
        try:
            record["confidence"] = min(1.0, max(0.0, float(record.get("confidence", 0.0))))
        except Exception:
            record["confidence"] = 0.0
        comparable = parse_bool(record.get("direct_causal_comparability"))
        record["direct_causal_comparability"] = bool(comparable) if comparable is not None else (record["evidence_quality"] in {"strong", "adequate"})
        fallback_policy, fallback_sentiment = default_directional_deficits(record["classification"])
        for key, fallback in [
            ("policy_to_sentiment_deficit", fallback_policy),
            ("sentiment_to_policy_deficit", fallback_sentiment),
        ]:
            try:
                value = float(record.get(key, fallback))
                record[key] = min(1.0, max(0.0, value)) if np.isfinite(value) else np.nan
            except Exception:
                record[key] = fallback
        findings.append(record)
    return findings


def finding_signature(finding: dict[str, Any]) -> str:
    parts = [
        finding.get("classification", ""), finding.get("relation_family", ""),
        finding.get("cause_theme", ""), finding.get("effect_theme", ""),
    ]
    return "::".join(re.sub(r"[^a-z0-9]+", " ", str(part).lower()).strip() for part in parts)


def coarse_profile(finding: dict[str, Any]) -> str:
    return f"{finding.get('classification','')}::{finding.get('relation_family','')}"


def json_cell(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False)


packages, empirical_evidence_df = build_empirical_packages(empirical)
robustness_packages, robustness_evidence_df = build_robustness_packages(packages, synthetic)
print("Empirical packages:", len(packages))
print("Synthetic robustness packages:", len(robustness_packages))
if packages:
    display(pd.DataFrame([{"analysis_id": p["analysis_id"], "scope": p["scope"], **p["evidence_counts"]} for p in packages]))

Empirical packages: 9
Synthetic robustness packages: 9


,analysis_id,scope,policy,sentiment,policy_sources,sentiment_sources,policy_causal_hints,sentiment_causal_hints
0,global__batch_01,global,24,24,23,17,21,21
1,global__batch_02,global,24,24,22,17,20,22
2,global__batch_03,global,24,24,22,17,21,20
3,france__batch_01,france,24,24,7,4,20,20
4,france__batch_02,france,24,24,7,4,22,21
5,france__batch_03,france,24,24,7,4,21,21
6,ireland__batch_01,ireland,24,24,15,3,21,21
7,ireland__batch_02,ireland,24,24,15,3,22,21
8,ireland__batch_03,ireland,24,24,15,3,21,23


In [4]:
SEMANTIC_ANALYSIS_PROMPT = r"""
You analyse semantic gaps between policy and sentiment using only the supplied cleaned source sentences.
The task concerns expressed causal language, not real-world causal effects.

For each potential finding, first determine whether the cited sentences contain a usable causal or
conditional relationship: a cause, prerequisite, risk, support, prevention, increase/decrease, or expected
outcome linked to an effect or outcome. Ignore sentences that are merely descriptive.

Critical comparison rules:
- Use no outside knowledge.
- Treat evidence IDs as immutable.
- The causal_hint field is only a sampling hint and is not evidence that a sentence is causal.
- Never infer a policy_gap or sentiment_gap merely because the opposite corpus lacks a matching sentence
  in this small evidence batch.
- A substantive comparison must cite directly comparable causal evidence from both corpora.
- "Same broad topic" is not enough. The cited sentences must address the same or a closely related cause,
  prerequisite, risk, support mechanism, or outcome.
- Descriptive percentages, headings, labels, or general statements are not causal evidence unless the
  sentence itself expresses the relationship used in the finding.
- If one side is only thematically related, return insufficient_evidence rather than constructing a gap.
- Compare meanings, not wording alone.
- Consider supporting and contradictory evidence.
- Alignment and insufficient_evidence are valid outcomes.
- Keep cause_theme and effect_theme to short, stable noun phrases, preferably 2--6 words.
- Use one of the supplied relation families when possible.
- Return at most the requested number of findings.

Relation families:
causes_or_increases, reduces_or_prevents, enables_or_supports,
requires_or_depends_on, risks_or_threatens, expected_to_improve, other.

Return one JSON object with exactly this structure:
{
  "analysis_id": "exact input analysis_id",
  "findings": [
    {
      "finding_id": "S1, S2, ...",
      "classification": "policy_gap | sentiment_gap | partial_alignment | alignment | insufficient_evidence",
      "relation_family": "one relation family",
      "cause_theme": "short cause or prerequisite theme",
      "effect_theme": "short effect or outcome theme",
      "gap_label": "short descriptive label",
      "explanation": "no more than 90 words",
      "policy_evidence_ids": ["exact policy evidence IDs"],
      "sentiment_evidence_ids": ["exact sentiment evidence IDs"],
      "counterevidence_ids": ["exact evidence IDs"],
      "direct_causal_comparability": true,
      "policy_to_sentiment_deficit": 0.0,
      "sentiment_to_policy_deficit": 0.0,
      "confidence": 0.0
    }
  ],
  "discarded_noncausal_ids": ["evidence IDs judged non-causal or unusable"],
  "limitations": ["short limitations"]
}

Directional deficit scores:
- policy_to_sentiment_deficit: 0 means the cited policy causal meaning is fully covered by the cited sentiment evidence; 1 means it is materially unsupported or different.
- sentiment_to_policy_deficit: the converse direction.
- Use the same anchors in both directions: 0.00--0.20 substantially corresponding; 0.21--0.40 mostly covered with minor differences; 0.41--0.60 partial coverage; 0.61--0.80 substantial mismatch; 0.81--1.00 very weak coverage among the directly comparable cited evidence.
- Score only the supplied cited evidence. Do not convert absence in a small batch into a high deficit.
- Report both directional scores for every reportable finding.

Definitions:
- policy_gap: directly comparable evidence exists in both corpora, but policy expresses a materially
  different or stronger causal emphasis than sentiment.
- sentiment_gap: directly comparable evidence exists in both corpora, but sentiment expresses a materially
  different or stronger causal emphasis than policy.
- partial_alignment: both corpora express related causal meaning but differ materially in emphasis,
  relation, cause, or outcome.
- alignment: the supplied evidence expresses substantially corresponding causal meaning.
- insufficient_evidence: the supplied evidence does not justify a reliable causal semantic comparison.
""".strip()

SEMANTIC_VERIFICATION_PROMPT = r"""
Independently verify the candidate findings using only the reordered evidence.

Be conservative. A finding is not evidence-grounded merely because both citations mention AI or the same
general topic. The cause/prerequisite and effect/outcome must be directly comparable.

Check:
- the analysis_id and every cited evidence ID;
- whether each cited sentence genuinely expresses the causal or conditional relationship attributed to it;
- whether both corpora directly support the comparison rather than one side being merely thematic;
- direct_causal_comparability must be true only when both sides express directly comparable causal or conditional meaning;
- omitted counterevidence;
- relation-family, cause-theme, and effect-theme consistency;
- whether the explanation follows from the cited sentences;
- whether batch-level absence is incorrectly presented as corpus-wide absence;
- whether policy_gap or sentiment_gap should instead be partial_alignment, alignment, or insufficient_evidence.

For every verified finding assign evidence_quality:
- strong: both corpora directly express the same or closely corresponding causal relationship;
- adequate: both corpora address the same causal relationship, but one side is less explicit or differs
  materially in cause, effect, or framing;
- weak: one side is only broadly thematic, descriptive, inferred from absence, or not genuinely causal.

Omit weak findings from verified_findings. Reclassify a candidate when the evidence supports a different
classification. Keep cause_theme and effect_theme short and canonical. Recalculate both directional deficit
scores independently from the cited evidence rather than copying the analyst scores. Use the same 0--1 anchors
specified for the analyst and never infer a high deficit from batch-level absence.

Return one JSON object with exactly this structure:
{
  "analysis_id": "exact input analysis_id",
  "verdict": "accept | revise | reject",
  "verified_findings": [
    {
      "finding_id": "S1, S2, ...",
      "classification": "policy_gap | sentiment_gap | partial_alignment | alignment | insufficient_evidence",
      "relation_family": "one relation family",
      "cause_theme": "short cause or prerequisite theme",
      "effect_theme": "short effect or outcome theme",
      "gap_label": "short descriptive label",
      "explanation": "no more than 90 words",
      "policy_evidence_ids": ["exact policy evidence IDs"],
      "sentiment_evidence_ids": ["exact sentiment evidence IDs"],
      "counterevidence_ids": ["exact evidence IDs"],
      "evidence_quality": "strong | adequate | weak",
      "direct_causal_comparability": true,
      "policy_to_sentiment_deficit": 0.0,
      "sentiment_to_policy_deficit": 0.0,
      "confidence": 0.0
    }
  ],
  "invalid_evidence_ids": ["IDs"],
  "unresolved_counterevidence_ids": ["IDs"],
  "evidence_faithfulness": 0.0,
  "reason": "no more than 70 words"
}

For each verified finding, also score the two directional semantic deficits from 0 to 1 using only the cited evidence. A high score means more unmatched or materially different causal meaning in that direction. These scores are descriptive agent judgements used for native-topic diagnostics, not cosine distances.

Use an empty verified_findings list when the candidate must be rejected.
""".strip()

ANALYSIS_KEYS = {"analysis_id", "findings", "discarded_noncausal_ids", "limitations"}
VERIFICATION_KEYS = {
    "analysis_id", "verdict", "verified_findings", "invalid_evidence_ids",
    "unresolved_counterevidence_ids", "evidence_faithfulness", "reason",
}


In [5]:
def run_key(analysis_id: str, run_index: int) -> tuple[str, int]:
    return analysis_id, run_index


def package_fingerprint(package: dict[str, Any]) -> str:
    payload = {
        "analysis_id": package["analysis_id"],
        "evidence": [
            {
                "evidence_id": item["evidence_id"],
                "corpus": item["corpus"],
                "text": item["text"],
            }
            for item in package["evidence"]
        ],
    }
    serialised = json.dumps(payload, ensure_ascii=False, sort_keys=True).encode("utf-8")
    return hashlib.sha256(serialised).hexdigest()


def load_previous_runs(path: Path) -> dict[tuple[str, int], dict[str, Any]]:
    previous={}
    if not RESUME_RUNS or not path.exists():
        return previous
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line=line.strip()
            if not line:
                continue
            try:
                row=json.loads(line)
                if row.get("pipeline_version") != PIPELINE_VERSION:
                    continue
                if row.get("prompt_version") != PROMPT_VERSION:
                    continue
                previous[run_key(str(row["analysis_id"]), int(row["run_index"]))] = row
            except Exception:
                continue
    return previous


def save_runs(path: Path, rows: Iterable[dict[str, Any]]) -> None:
    ordered=sorted(rows, key=lambda row: (str(row.get("analysis_id", "")), int(row.get("run_index", 0))))
    with path.open("w", encoding="utf-8") as handle:
        for row in ordered:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def validate_verified_findings(
    package: dict[str, Any],
    verification: dict[str, Any],
) -> tuple[list[dict[str, Any]], list[dict[str, str]]]:
    """Validate findings independently.

    A malformed or low-confidence finding is removed without invalidating other
    verified findings from the same run.
    """
    findings=normalise_findings(verification.get("verified_findings", []))
    all_ids=valid_ids(package)
    policy_ids=valid_ids(package, "policy")
    sentiment_ids=valid_ids(package, "sentiment")
    valid_findings=[]
    issues=[]

    for finding in findings:
        finding_id=finding["finding_id"]
        cited_policy=set(finding["policy_evidence_ids"])
        cited_sentiment=set(finding["sentiment_evidence_ids"])
        cited_counter=set(finding["counterevidence_ids"])

        evidence_is_valid=(
            cited_policy.issubset(policy_ids)
            and cited_sentiment.issubset(sentiment_ids)
            and cited_counter.issubset(all_ids)
        )
        reportable=finding["classification"] in REPORTABLE_CLASSIFICATIONS
        both_corpora=bool(cited_policy) and bool(cited_sentiment)

        reason=None
        if not evidence_is_valid:
            reason="invalid evidence ID"
        elif reportable and not both_corpora:
            reason="reportable finding lacks evidence from both corpora"
        elif reportable and (not finding["cause_theme"] or not finding["effect_theme"]):
            reason="missing cause/effect theme"
        elif reportable and finding["evidence_quality"] == "weak":
            reason="verifier marked evidence weak"
        elif reportable and REQUIRE_DIRECT_COMPARABILITY and not finding.get("direct_causal_comparability", False):
            reason="verifier did not confirm direct causal comparability"
        elif reportable and not (
            np.isfinite(finding.get("policy_to_sentiment_deficit", np.nan))
            and np.isfinite(finding.get("sentiment_to_policy_deficit", np.nan))
        ):
            reason="missing verifier directional deficit score"
        elif finding["confidence"] < MIN_CONFIDENCE:
            reason="confidence below threshold"

        if reason:
            issues.append({"finding_id": finding_id, "reason": reason})
            continue
        valid_findings.append(finding)

    return valid_findings, issues


def run_packages(package_list: list[dict[str, Any]], runs_path: Path, seed_offset: int = 0) -> list[dict[str, Any]]:
    """Run until the target number of usable analyst/verifier repetitions is reached.

    A failed or unusable attempt does not consume a stability replicate. Attempts are
    capped so a problematic package cannot loop indefinitely. Every attempt is cached.
    """
    previous = load_previous_runs(runs_path)
    rows=[]

    for package_index, package in enumerate(package_list):
        analysis_id=package["analysis_id"]
        fingerprint=package_fingerprint(package)
        usable_count=0

        for zero_based_attempt in range(MAX_RUN_ATTEMPTS):
            if usable_count >= TARGET_USABLE_RUNS:
                break

            run_index=zero_based_attempt+1
            key=run_key(analysis_id, run_index)
            cached=previous.get(key)
            if cached is not None and cached.get("package_fingerprint") == fingerprint:
                rows.append(cached)
                usable_count += int(bool(cached.get("run_usable")))
                continue

            if RUN_AGENT:
                candidate=ask_agent(
                    SEMANTIC_ANALYSIS_PROMPT,
                    shuffled_package(package, RANDOM_STATE + seed_offset + package_index*100 + zero_based_attempt),
                    action=f"Analysing {analysis_id}, attempt {run_index}",
                    required_keys=ANALYSIS_KEYS,
                    temperature=0.0,
                )
                verification=ask_agent(
                    SEMANTIC_VERIFICATION_PROMPT,
                    {
                        "analysis_id": analysis_id,
                        "candidate": candidate,
                        "case": shuffled_package(
                            package,
                            RANDOM_STATE + seed_offset + 10000 + package_index*100 + zero_based_attempt,
                        ),
                    },
                    action=f"Verifying {analysis_id}, attempt {run_index}",
                    required_keys=VERIFICATION_KEYS,
                    temperature=0.0,
                )
            else:
                candidate={
                    "analysis_id": analysis_id,
                    "findings": [],
                    "discarded_noncausal_ids": [],
                    "limitations": ["Agent execution is disabled."],
                    "status": "not_run",
                }
                verification={
                    "analysis_id": analysis_id,
                    "verdict": "reject",
                    "verified_findings": [],
                    "invalid_evidence_ids": [],
                    "unresolved_counterevidence_ids": [],
                    "evidence_faithfulness": 0.0,
                    "reason": "Configure the agent and enable RUN_DEEPSEEK_AGENT.",
                    "status": "not_run",
                }

            try:
                faithfulness=min(1.0, max(0.0, float(verification.get("evidence_faithfulness", 0.0))))
            except Exception:
                faithfulness=0.0

            verdict=str(verification.get("verdict", "reject")).strip().lower()
            id_matches=(
                str(candidate.get("analysis_id", "")) == analysis_id
                and str(verification.get("analysis_id", "")) == analysis_id
            )
            invalid_ids=normalise_ids(verification.get("invalid_evidence_ids", []))
            unresolved_ids=normalise_ids(verification.get("unresolved_counterevidence_ids", []))
            valid_findings, finding_issues=validate_verified_findings(package, verification)

            run_usable=bool(
                id_matches
                and verdict in {"accept", "revise"}
                and faithfulness >= MIN_FAITHFULNESS
                and valid_findings
            )

            row={
                "pipeline_version": PIPELINE_VERSION,
                "prompt_version": PROMPT_VERSION,
                "package_fingerprint": fingerprint,
                "analysis_id": analysis_id,
                "scope": package["scope"],
                "batch_id": package["batch_id"],
                "run_index": run_index,
                "candidate": candidate,
                "verification": verification,
                "findings": valid_findings,
                "verdict": verdict,
                "evidence_faithfulness": faithfulness,
                "analysis_id_matches": id_matches,
                "invalid_evidence_ids": invalid_ids,
                "unresolved_counterevidence_ids": unresolved_ids,
                "finding_issues": finding_issues,
                "run_usable": run_usable,
                "analysis_model": candidate.get("_agent", {}).get("model"),
                "verification_model": verification.get("_agent", {}).get("model"),
            }
            if "probe_of_analysis_id" in package:
                row["probe_of_analysis_id"] = package["probe_of_analysis_id"]

            rows.append(row)
            previous[key]=row
            usable_count += int(run_usable)
            save_runs(runs_path, previous.values())

    return rows


empirical_run_rows = run_packages(packages, RUNS_PATH, seed_offset=0)
robustness_run_rows = (
    run_packages(robustness_packages, ROBUSTNESS_RUNS_PATH, seed_offset=30000)
    if robustness_packages
    else []
)

print("Empirical run records:", len(empirical_run_rows))
print("Usable empirical runs:", sum(bool(row.get("run_usable")) for row in empirical_run_rows))
print("Robustness run records:", len(robustness_run_rows))
print("Usable robustness runs:", sum(bool(row.get("run_usable")) for row in robustness_run_rows))


Analysing global__batch_01, attempt 1 (deepseek-v4-pro, schema attempt 1/2)...
Verifying global__batch_01, attempt 1 (deepseek-v4-pro, schema attempt 1/2)...
Analysing global__batch_01, attempt 2 (deepseek-v4-pro, schema attempt 1/2)...
Verifying global__batch_01, attempt 2 (deepseek-v4-pro, schema attempt 1/2)...
Analysing global__batch_01, attempt 3 (deepseek-v4-pro, schema attempt 1/2)...
Verifying global__batch_01, attempt 3 (deepseek-v4-pro, schema attempt 1/2)...
Analysing global__batch_01, attempt 4 (deepseek-v4-pro, schema attempt 1/2)...
Verifying global__batch_01, attempt 4 (deepseek-v4-pro, schema attempt 1/2)...
Analysing global__batch_02, attempt 1 (deepseek-v4-pro, schema attempt 1/2)...
Verifying global__batch_02, attempt 1 (deepseek-v4-pro, schema attempt 1/2)...
Analysing global__batch_02, attempt 2 (deepseek-v4-pro, schema attempt 1/2)...
Verifying global__batch_02, attempt 2 (deepseek-v4-pro, schema attempt 1/2)...
Analysing global__batch_02, attempt 3 (deepseek-v4-p

In [6]:
MATCH_STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "to", "in", "on", "for", "with", "by",
    "from", "as", "at", "into", "through", "ai", "use", "using", "used",
    "artificial", "intelligence", "genai",
}


def theme_text(finding: dict[str, Any]) -> str:
    value = " ".join([
        str(finding.get("cause_theme", "")),
        str(finding.get("effect_theme", "")),
        str(finding.get("gap_label", "")),
    ]).lower()
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", value)).strip()


def theme_tokens(finding: dict[str, Any]) -> set[str]:
    return {
        token
        for token in re.findall(r"[a-z0-9]+", theme_text(finding))
        if len(token) > 2 and token not in MATCH_STOPWORDS
    }


def set_jaccard(left: set[str], right: set[str]) -> float:
    union=left | right
    return 1.0 if not union else len(left & right) / len(union)


def finding_match_score(left: dict[str, Any], right: dict[str, Any]) -> float:
    """Match recurrent findings without requiring identical free-text labels."""
    classification_agrees = left.get("classification") == right.get("classification")
    relation_agrees = left.get("relation_family") == right.get("relation_family")

    left_tokens=theme_tokens(left)
    right_tokens=theme_tokens(right)
    token_similarity=set_jaccard(left_tokens, right_tokens)
    sequence_similarity=SequenceMatcher(None, theme_text(left), theme_text(right)).ratio()
    theme_similarity=max(token_similarity, 0.75 * sequence_similarity)

    policy_similarity=set_jaccard(
        set(left.get("policy_evidence_ids", [])),
        set(right.get("policy_evidence_ids", [])),
    )
    sentiment_similarity=set_jaccard(
        set(left.get("sentiment_evidence_ids", [])),
        set(right.get("sentiment_evidence_ids", [])),
    )
    evidence_similarity=0.5 * (policy_similarity + sentiment_similarity)

    # Match the underlying causal comparison before testing classification stability.
    # Classification therefore does not contribute to identity. Relation agreement is
    # only a small semantic cue; evidence and cause/effect themes dominate.
    if theme_similarity <= 0.0 and evidence_similarity <= 0.0:
        return 0.0
    return float(
        0.50 * theme_similarity
        + 0.40 * evidence_similarity
        + 0.10 * float(relation_agrees)
    )


def cluster_recurrent_findings(run_group: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Greedy one-to-one matching across runs, ordered by strongest match first."""
    usable=sorted(
        [row for row in run_group if row.get("run_usable")],
        key=lambda row: int(row["run_index"]),
    )
    clusters: list[dict[str, Any]]=[]

    for row in usable:
        run_index=int(row["run_index"])
        findings=list(row.get("findings", []))
        if not clusters:
            for finding in findings:
                clusters.append({"members": [(run_index, row["evidence_faithfulness"], finding)]})
            continue

        edges=[]
        for finding_index, finding in enumerate(findings):
            for cluster_index, cluster in enumerate(clusters):
                if any(member_run == run_index for member_run, _, _ in cluster["members"]):
                    continue
                score=max(
                    finding_match_score(finding, member)
                    for _, _, member in cluster["members"]
                )
                edges.append((score, finding_index, cluster_index))

        used_findings=set()
        used_clusters=set()
        for score, finding_index, cluster_index in sorted(edges, reverse=True):
            if score < MIN_MATCH_SCORE:
                break
            if finding_index in used_findings or cluster_index in used_clusters:
                continue
            clusters[cluster_index]["members"].append(
                (run_index, row["evidence_faithfulness"], findings[finding_index])
            )
            used_findings.add(finding_index)
            used_clusters.add(cluster_index)

        for finding_index, finding in enumerate(findings):
            if finding_index not in used_findings:
                clusters.append({"members": [(run_index, row["evidence_faithfulness"], finding)]})

    return clusters


def pairwise_mean(values: list[Any], score_fn) -> float:
    if len(values) < 2:
        return 0.0
    scores=[score_fn(left, right) for left, right in itertools.combinations(values, 2)]
    return float(np.mean(scores)) if scores else 0.0


def modal_value(values: list[str]) -> tuple[str, float]:
    clean = [str(value) for value in values if str(value)]
    if not clean:
        return "", 0.0
    counts = Counter(clean)
    value, count = counts.most_common(1)[0]
    return value, count / len(clean)


def summarise_empirical_runs(
    package_list: list[dict[str, Any]],
    run_rows: list[dict[str, Any]],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    stability_rows=[]
    candidate_rows=[]
    stable_counter=0

    for package in package_list:
        analysis_id=package["analysis_id"]
        group=sorted(
            [row for row in run_rows if row["analysis_id"] == analysis_id],
            key=lambda row: row["run_index"],
        )
        usable=[row for row in group if row.get("run_usable")]
        clusters=cluster_recurrent_findings(group)

        retained_clusters=0
        for cluster_index, cluster in enumerate(clusters, start=1):
            members=cluster["members"]
            run_indices=sorted({run_index for run_index, _, _ in members})
            findings=[finding for _, _, finding in members]
            faithfulness_values=[faith for _, faith, _ in members]
            recurrence=len(run_indices)

            match_stability=pairwise_mean(findings, finding_match_score)
            evidence_sets=[
                set(finding["policy_evidence_ids"] + finding["sentiment_evidence_ids"])
                for finding in findings
            ]
            evidence_stability=pairwise_mean(evidence_sets, set_jaccard)
            mean_confidence=float(np.mean([finding["confidence"] for finding in findings]))
            mean_faithfulness=float(np.mean(faithfulness_values))
            modal_classification, classification_agreement = modal_value([finding["classification"] for finding in findings])
            modal_relation, relation_agreement = modal_value([finding["relation_family"] for finding in findings])
            direct_comparability_share = float(np.mean([
                bool(finding.get("direct_causal_comparability", False)) for finding in findings
            ]))

            # Select the strongest member among those supporting the modal classification when possible.
            representative_pool = [
                member for member in members if member[2]["classification"] == modal_classification
            ] or members
            representative=max(
                representative_pool,
                key=lambda item: (
                    item[2]["evidence_quality"] == "strong",
                    item[2]["confidence"],
                    item[1],
                    len(item[2]["policy_evidence_ids"]) + len(item[2]["sentiment_evidence_ids"]),
                ),
            )[2]

            machine_retained=bool(
                recurrence >= MIN_RECURRENT_RUNS
                and match_stability >= MIN_MATCH_SCORE
                and mean_confidence >= MIN_CONFIDENCE
                and mean_faithfulness >= MIN_FAITHFULNESS
                and classification_agreement >= MIN_CLASSIFICATION_AGREEMENT
                and relation_agreement >= MIN_RELATION_AGREEMENT
                and direct_comparability_share >= MIN_DIRECT_COMPARABILITY_SHARE
                and representative["evidence_quality"] in {"strong", "adequate"}
                and representative.get("direct_causal_comparability", False)
                and modal_classification in REPORTABLE_CLASSIFICATIONS
            )
            if machine_retained:
                retained_clusters += 1
                stable_counter += 1

            candidate_rows.append({
                "analysis_id": analysis_id,
                "scope": package["scope"],
                "batch_id": package["batch_id"],
                "cluster_id": f"{analysis_id}__cluster_{cluster_index:02d}",
                "finding_id": representative["finding_id"],
                "classification": modal_classification,
                "relation_family": modal_relation or representative["relation_family"],
                "cause_theme": representative["cause_theme"],
                "effect_theme": representative["effect_theme"],
                "gap_label": representative["gap_label"],
                "explanation": representative["explanation"],
                "policy_evidence_ids": json_cell(representative["policy_evidence_ids"]),
                "sentiment_evidence_ids": json_cell(representative["sentiment_evidence_ids"]),
                "counterevidence_ids": json_cell(representative["counterevidence_ids"]),
                "evidence_quality": representative["evidence_quality"],
                "direct_causal_comparability": representative.get("direct_causal_comparability", False),
                "direct_comparability_share": direct_comparability_share,
                "policy_to_sentiment_deficit": float(np.mean([f["policy_to_sentiment_deficit"] for f in findings if np.isfinite(f.get("policy_to_sentiment_deficit", np.nan))])) if any(np.isfinite(f.get("policy_to_sentiment_deficit", np.nan)) for f in findings) else np.nan,
                "sentiment_to_policy_deficit": float(np.mean([f["sentiment_to_policy_deficit"] for f in findings if np.isfinite(f.get("sentiment_to_policy_deficit", np.nan))])) if any(np.isfinite(f.get("sentiment_to_policy_deficit", np.nan)) for f in findings) else np.nan,
                "classification_agreement": classification_agreement,
                "relation_agreement": relation_agreement,
                "representative_confidence": representative["confidence"],
                "mean_confidence": mean_confidence,
                "mean_evidence_faithfulness": mean_faithfulness,
                "recurrent_runs": recurrence,
                "total_runs": len(group),
                "usable_runs": len(usable),
                "run_recurrence": recurrence / max(1, len(usable)),
                "attempt_recurrence": recurrence / max(1, len(group)),
                "finding_match_stability": match_stability,
                "evidence_jaccard_stability": evidence_stability,
                "member_finding_ids": json_cell([
                    f"run_{run_index}:{finding['finding_id']}"
                    for run_index, _, finding in members
                ]),
                "machine_retained": machine_retained,
            })

        stability_rows.append({
            "analysis_id": analysis_id,
            "scope": package["scope"],
            "batch_id": package["batch_id"],
            "total_runs": len(group),
            "usable_runs": len(usable),
            "usable_run_rate": len(usable) / max(1, len(group)),
            "validated_findings": int(sum(len(row.get("findings", [])) for row in usable)),
            "finding_clusters": len(clusters),
            "retained_recurrent_clusters": retained_clusters,
        })

    stability_df=pd.DataFrame(stability_rows)
    candidates_df=pd.DataFrame(candidate_rows)
    return stability_df, candidates_df


stability_df, candidates_df = summarise_empirical_runs(packages, empirical_run_rows)
findings_df = (
    candidates_df[candidates_df["machine_retained"].eq(True)].copy()
    if not candidates_df.empty
    else pd.DataFrame()
)

write_nonempty_csv(stability_df, STABILITY_PATH)
write_nonempty_csv(candidates_df, CANDIDATES_PATH)
write_nonempty_csv(findings_df, FINDINGS_PATH)

# Preserve finding-level validation issues for auditing without invalidating whole runs.
issue_rows=[]
for row in empirical_run_rows:
    for issue in row.get("finding_issues", []):
        issue_rows.append({
            "analysis_id": row["analysis_id"],
            "scope": row["scope"],
            "batch_id": row["batch_id"],
            "run_index": row["run_index"],
            "finding_id": issue.get("finding_id", ""),
            "reason": issue.get("reason", ""),
            "run_usable": row.get("run_usable", False),
            "evidence_faithfulness": row.get("evidence_faithfulness", 0.0),
        })
    for evidence_id in row.get("invalid_evidence_ids", []):
        issue_rows.append({
            "analysis_id": row["analysis_id"],
            "scope": row["scope"],
            "batch_id": row["batch_id"],
            "run_index": row["run_index"],
            "finding_id": "",
            "reason": f"verifier reported invalid evidence ID: {evidence_id}",
            "run_usable": row.get("run_usable", False),
            "evidence_faithfulness": row.get("evidence_faithfulness", 0.0),
        })
write_nonempty_csv(pd.DataFrame(issue_rows), RUN_ISSUES_PATH)

# Traceable evidence table for retained empirical findings only.
evidence_lookup = empirical.set_index("sentence_id", drop=False)
finding_evidence_rows=[]
for _, finding in findings_df.iterrows():
    for role, column in [
        ("policy", "policy_evidence_ids"),
        ("sentiment", "sentiment_evidence_ids"),
        ("counter", "counterevidence_ids"),
    ]:
        try:
            ids=json.loads(finding[column])
        except Exception:
            ids=[]
        for evidence_id in ids:
            if evidence_id not in evidence_lookup.index:
                continue
            row=evidence_lookup.loc[evidence_id]
            finding_evidence_rows.append({
                "analysis_id": finding["analysis_id"],
                "cluster_id": finding["cluster_id"],
                "finding_id": finding["finding_id"],
                "classification": finding["classification"],
                "relation_family": finding["relation_family"],
                "evidence_role": role,
                "evidence_id": evidence_id,
                "corpus": row["corpus"],
                "text": row["clean_sentence"],
                "source_file": row["source_file"],
                "doc_id": row["doc_id"],
                "country": row["analysis_country"],
            })
finding_evidence_df=pd.DataFrame(finding_evidence_rows)
write_nonempty_csv(finding_evidence_df, FINDING_EVIDENCE_PATH)

print("Validated candidate clusters:", len(candidates_df))
print("Retained recurrent empirical findings:", len(findings_df))
display(stability_df)
if not findings_df.empty:
    display(findings_df.head(20))


Validated candidate clusters: 58
Retained recurrent empirical findings: 16


,analysis_id,scope,batch_id,total_runs,usable_runs,usable_run_rate,validated_findings,finding_clusters,retained_recurrent_clusters
0,global__batch_01,global,1,4,3,0.75,10,6,2
1,global__batch_02,global,2,3,3,1.00,10,6,1
2,global__batch_03,global,3,4,3,0.75,10,7,1
3,france__batch_01,france,1,3,3,1.00,4,3,1
4,france__batch_02,france,2,3,3,1.00,15,10,4
5,france__batch_03,france,3,4,3,0.75,11,7,3
6,ireland__batch_01,ireland,1,4,3,0.75,11,7,2
7,ireland__batch_02,ireland,2,4,3,0.75,9,6,1
8,ireland__batch_03,ireland,3,3,3,1.00,8,6,1


,analysis_id,scope,batch_id,cluster_id,finding_id,classification,relation_family,cause_theme,effect_theme,gap_label,explanation,policy_evidence_ids,sentiment_evidence_ids,counterevidence_ids,evidence_quality,direct_causal_comparability,direct_comparability_share,policy_to_sentiment_deficit,sentiment_to_policy_deficit,classification_agreement,relation_agreement,representative_confidence,mean_confidence,mean_evidence_faithfulness,recurrent_runs,total_runs,usable_runs,run_recurrence,attempt_recurrence,finding_match_stability,evidence_jaccard_stability,member_finding_ids,machine_retained
0,global__batch_01,global,1,global__batch_01__cluster_01,S1,partial_alignment,enables_or_supports,AI tools,teacher communication,AI support for teacher communication,Policy evidence states AI can assist in writin...,"[""clean_sentence_005462""]","[""clean_sentence_018986""]",[],adequate,True,1.0,0.400000,0.400000,1.000000,1.000000,0.70,0.700000,0.833333,3,4,3,1.000000,0.750000,0.627824,0.444444,"[""run_1:S1"", ""run_2:S1"", ""run_4:S1""]",True
3,global__batch_01,global,1,global__batch_01__cluster_04,S3,partial_alignment,requires_or_depends_on,teacher training,effective AI use,Teacher training for AI use,Policy evidence indicates teachers are expecte...,"[""clean_sentence_002786""]","[""clean_sentence_019187""]",[],adequate,True,1.0,0.350000,0.350000,1.000000,1.000000,0.70,0.700000,0.850000,2,4,3,0.666667,0.500000,0.875000,1.000000,"[""run_1:S4"", ""run_2:S3""]",True
6,global__batch_02,global,2,global__batch_02__cluster_01,S1,partial_alignment,risks_or_threatens,AI-assisted learning,critical thinking,AI reduces critical thinking,Policy evidence states AI can enhance decision...,"[""clean_sentence_008623""]","[""clean_sentence_017817""]",[],adequate,True,1.0,0.666667,0.666667,1.000000,0.666667,0.80,0.800000,0.833333,3,3,3,1.000000,1.000000,0.877778,1.000000,"[""run_1:S1"", ""run_2:S1"", ""run_3:S1""]",True
17,global__batch_03,global,3,global__batch_03__cluster_06,S3,alignment,enables_or_supports,AI tools,teaching and learning support,AI support for education,Policy evidence states generative AI has poten...,"[""clean_sentence_009416""]","[""clean_sentence_018320""]",[],strong,True,1.0,0.150000,0.150000,1.000000,1.000000,0.90,0.850000,0.850000,2,4,3,0.666667,0.500000,0.670690,1.000000,"[""run_3:S3"", ""run_4:S3""]",True
19,france__batch_01,france,1,france__batch_01__cluster_01,S1,partial_alignment,enables_or_supports,AI training,AI usage,Training impact on AI usage,Policy states training had limited impact on A...,"[""clean_sentence_005123""]","[""clean_sentence_018632""]",[],adequate,True,1.0,0.600000,0.600000,1.000000,1.000000,0.70,0.700000,0.800000,2,3,3,0.666667,0.666667,1.000000,1.000000,"[""run_1:S1"", ""run_2:S1""]",True
23,france__batch_02,france,2,france__batch_02__cluster_02,S4,partial_alignment,risks_or_threatens,AI use,ethical risks,Ethical risks of AI acknowledged in both polic...,Policy sentence 011254 lists ethical risks of ...,"[""clean_sentence_011254""]","[""clean_sentence_018929""]",[],strong,True,1.0,0.433333,0.433333,1.000000,1.000000,0.80,0.766667,0.900000,3,3,3,1.000000,1.000000,0.759642,1.000000,"[""run_1:S2"", ""run_2:S1"", ""run_3:S4""]",True
24,france__batch_02,france,2,france__batch_02__cluster_03,S3,partial_alignment,enables_or_supports,AI training,AI understanding,Training for AI literacy,Policy emphasizes school's role in supporting ...,"[""clean_sentence_012375"", ""clean_sentence_0127...","[""clean_sentence_015925""]",[],adequate,True,1.0,0.500000,0.500000,1.000000,1.000000,0.70,0.700000,0.900000,2,3,3,0.666667,0.666667,0.733333,0.666667,"[""run_1:S3"", ""run_2:S5""]",True
27,france__batch_02,france,2,france__batch_02__cluster_06,S2,partial_alignment,enables_or_supports,digital mediation center,awareness of digital help,Digital mediation boosts awareness,Sentiment sentence 017896 states living near a...,"[""clean_sentence_012375""]","[""clean_sentence_017896""]",[],adequate,True,1.0,0.500000,0.500000,1.000000,1.000000

In [7]:
def distribution_tvd(left: list[str], right: list[str], labels: list[str]) -> float:
    if not left or not right:
        return np.nan
    left_counts=Counter(left)
    right_counts=Counter(right)
    left_total=sum(left_counts.values())
    right_total=sum(right_counts.values())
    return 0.5 * sum(
        abs(left_counts.get(label, 0) / left_total - right_counts.get(label, 0) / right_total)
        for label in labels
    )


def validated_profile(run_rows: list[dict[str, Any]], analysis_id: str) -> dict[str, Any] | None:
    usable=[
        row for row in run_rows
        if row["analysis_id"] == analysis_id and row.get("run_usable")
    ]
    if not usable:
        return None

    findings=[
        finding
        for row in usable
        for finding in row.get("findings", [])
        if finding.get("classification") in REPORTABLE_CLASSIFICATIONS
    ]
    if not findings:
        return None

    return {
        "findings": findings,
        "classes": [finding["classification"] for finding in findings],
        "relations": [finding["relation_family"] for finding in findings],
        "profiles": {
            f"{finding['classification']}::{finding['relation_family']}"
            for finding in findings
        },
        "mean_confidence": float(np.mean([finding["confidence"] for finding in findings])),
        "mean_faithfulness": float(np.mean([row["evidence_faithfulness"] for row in usable])),
        "mean_policy_to_sentiment_deficit": float(np.nanmean([finding.get("policy_to_sentiment_deficit", np.nan) for finding in findings])),
        "mean_sentiment_to_policy_deficit": float(np.nanmean([finding.get("sentiment_to_policy_deficit", np.nan) for finding in findings])),
        "usable_runs": len(usable),
    }


robustness_rows=[]
for probe in robustness_packages:
    probe_id=probe["analysis_id"]
    base_id=probe["probe_of_analysis_id"]
    base=validated_profile(empirical_run_rows, base_id)
    perturbed=validated_profile(robustness_run_rows, probe_id)

    if base is None or perturbed is None:
        robustness_rows.append({
            "analysis_id": base_id,
            "probe_analysis_id": probe_id,
            "scope": probe["scope"],
            "batch_id": probe["batch_id"],
            "status": "not_available",
            "reason": "Usable validated findings were unavailable for the baseline or perturbation run.",
            "classification_tvd": np.nan,
            "relation_tvd": np.nan,
            "profile_jaccard": np.nan,
            "mean_confidence_change": np.nan,
            "faithfulness_change": np.nan,
            "policy_to_sentiment_deficit_change": np.nan,
            "sentiment_to_policy_deficit_change": np.nan,
            "baseline_validated_findings": 0 if base is None else len(base["findings"]),
            "probe_validated_findings": 0 if perturbed is None else len(perturbed["findings"]),
            "baseline_usable_runs": 0 if base is None else base["usable_runs"],
            "probe_usable_runs": 0 if perturbed is None else perturbed["usable_runs"],
        })
        continue

    robustness_rows.append({
        "analysis_id": base_id,
        "probe_analysis_id": probe_id,
        "scope": probe["scope"],
        "batch_id": probe["batch_id"],
        "status": "available",
        "reason": "",
        "classification_tvd": distribution_tvd(
            base["classes"], perturbed["classes"], sorted(REPORTABLE_CLASSIFICATIONS)
        ),
        "relation_tvd": distribution_tvd(
            base["relations"], perturbed["relations"], sorted(ALLOWED_RELATIONS)
        ),
        "profile_jaccard": set_jaccard(base["profiles"], perturbed["profiles"]),
        "mean_confidence_change": perturbed["mean_confidence"] - base["mean_confidence"],
        "faithfulness_change": perturbed["mean_faithfulness"] - base["mean_faithfulness"],
        "policy_to_sentiment_deficit_change": perturbed["mean_policy_to_sentiment_deficit"] - base["mean_policy_to_sentiment_deficit"],
        "sentiment_to_policy_deficit_change": perturbed["mean_sentiment_to_policy_deficit"] - base["mean_sentiment_to_policy_deficit"],
        "baseline_validated_findings": len(base["findings"]),
        "probe_validated_findings": len(perturbed["findings"]),
        "baseline_usable_runs": base["usable_runs"],
        "probe_usable_runs": perturbed["usable_runs"],
    })

robustness_summary_df=pd.DataFrame(robustness_rows)
write_nonempty_csv(robustness_summary_df, ROBUSTNESS_SUMMARY_PATH)

available_robustness = robustness_summary_df[robustness_summary_df["status"].eq("available")].copy() if not robustness_summary_df.empty else pd.DataFrame()
if not available_robustness.empty:
    available_robustness["profile_change"] = 1.0 - available_robustness["profile_jaccard"]
    robustness_scope_summary_df = available_robustness.groupby("scope", as_index=False).agg(
        batches=("analysis_id", "count"),
        mean_classification_tvd=("classification_tvd", "mean"),
        mean_relation_tvd=("relation_tvd", "mean"),
        mean_profile_change=("profile_change", "mean"),
        mean_confidence_change=("mean_confidence_change", "mean"),
        mean_faithfulness_change=("faithfulness_change", "mean"),
        mean_policy_to_sentiment_deficit_change=("policy_to_sentiment_deficit_change", "mean"),
        mean_sentiment_to_policy_deficit_change=("sentiment_to_policy_deficit_change", "mean"),
    )
else:
    robustness_scope_summary_df = pd.DataFrame()
write_nonempty_csv(robustness_scope_summary_df, ROBUSTNESS_SCOPE_SUMMARY_PATH)

print("Synthetic data are used only as a robustness perturbation.")
print("Synthetic findings are not exported as substantive candidate/finding tables.")
if not robustness_summary_df.empty:
    display(robustness_summary_df)


Synthetic data are used only as a robustness perturbation.
Synthetic findings are not exported as substantive candidate/finding tables.


,analysis_id,probe_analysis_id,scope,batch_id,status,reason,classification_tvd,relation_tvd,profile_jaccard,mean_confidence_change,faithfulness_change,policy_to_sentiment_deficit_change,sentiment_to_policy_deficit_change,baseline_validated_findings,probe_validated_findings,baseline_usable_runs,probe_usable_runs
0,global__batch_01,global__batch_01__synthetic_probe,global,1,available,,0.285714,0.514286,0.333333,0.118571,0.016667,-0.077143,-0.134286,10,7,3,3
1,global__batch_02,global__batch_02__synthetic_probe,global,2,available,,0.300000,0.318182,0.428571,-0.073636,-0.033333,-0.064545,-0.083636,10,11,3,3
2,global__batch_03,global__batch_03__synthetic_probe,global,3,available,,0.200000,0.316667,0.333333,-0.020833,0.000000,0.035000,0.091667,10,12,3,3
3,france__batch_01,france__batch_01__synthetic_probe,france,1,available,,0.150000,0.500000,0.200000,0.050000,0.016667,-0.175000,-0.175000,4,10,3,3
4,france__batch_02,france__batch_02__synthetic_probe,france,2,available,,0.000000,0.200000,0.750000,-0.013333,-0.066667,0.015000,-0.058333,15,12,3,3
5,france__batch_03,france__batch_03__synthetic_probe,france,3,available,,0.250000,0.219697,0.428571,0.109091,-0.033333,-0.166667,-0.130303,11,12,3,3
6,ireland__batch_01,ireland__batch_01__synthetic_probe,ireland,1,available,,0.643939,0.492424,0.428571,-0.023485,-0.016667,0.187121,-0.189394,11,12,3,3
7,ireland__batch_02,ireland__batch_02__synthetic_probe,ireland,2,available,,0.191919,0.343434,0.272727,-0.058081,-0.033333,0.142424,0.079798,9,11,3,3
8,ireland__batch_03,ireland__batch_03__synthetic_probe,ireland,3,available,,0.425000,0.150000,0.222222,0.007500,0.033333,0.080000,0.077500,8,10,3,3


In [8]:
HUMAN_COLUMNS = [
    "human_is_gap", "human_category", "human_evidence_faithful", "human_confirmed", "human_notes",
]
KEY_COLUMNS = ["analysis_id", "cluster_id"]


def build_review_table(candidates: pd.DataFrame) -> pd.DataFrame:
    review=candidates.copy()
    if review.empty:
        return review
    if REVIEW_PATH.exists():
        previous=pd.read_csv(REVIEW_PATH).fillna("")
        available=[column for column in HUMAN_COLUMNS if column in previous.columns]
        if all(column in previous.columns for column in KEY_COLUMNS):
            previous_labels=previous[KEY_COLUMNS+available].drop_duplicates(KEY_COLUMNS)
            review=review.merge(previous_labels, on=KEY_COLUMNS, how="left")
    for column in HUMAN_COLUMNS:
        if column not in review.columns:
            review[column]=""
        else:
            review[column]=review[column].fillna("")
    return review


review_df=build_review_table(candidates_df)
write_nonempty_csv(review_df, REVIEW_PATH)


def evaluate_human_review(review: pd.DataFrame) -> pd.DataFrame:
    if review.empty:
        return pd.DataFrame([{
            "labelled_findings": 0,
            "true_positives": 0,
            "false_positives": 0,
            "false_negatives": 0,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "status": "no agent candidates available for human review",
        }])

    if "human_is_gap" not in review.columns:
        raise ValueError("human_is_gap is missing from the review table.")

    labelled_table=review.copy()
    labelled_table["human_label"]=labelled_table["human_is_gap"].map(parse_bool)
    labelled=labelled_table[labelled_table["human_label"].notna()].copy()

    if labelled.empty:
        return pd.DataFrame([{
            "labelled_findings": 0,
            "true_positives": 0,
            "false_positives": 0,
            "false_negatives": 0,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "status": "enter yes/no labels in human_is_gap",
        }])

    labelled["machine_label"]=labelled["machine_retained"].map(parse_bool).fillna(False).astype(bool)
    labelled["human_label"]=labelled["human_label"].astype(bool)

    tp=int((labelled["machine_label"] & labelled["human_label"]).sum())
    fp=int((labelled["machine_label"] & ~labelled["human_label"]).sum())
    fn=int((~labelled["machine_label"] & labelled["human_label"]).sum())
    precision=tp/(tp+fp) if tp+fp else 0.0
    recall=tp/(tp+fn) if tp+fn else 0.0
    f1=2*precision*recall/(precision+recall) if precision+recall else 0.0

    return pd.DataFrame([{
        "labelled_findings": len(labelled),
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "status": "complete",
    }])


evaluation_df=evaluate_human_review(review_df)
evaluation_df.to_csv(EVALUATION_PATH, index=False)

if not review_df.empty:
    print("Human-review file:", REVIEW_PATH)
    display(review_df.head(20))
display(evaluation_df)


Human-review file: /home/nsirim/Github/mscdsa/msc/progress/causal_nlp/output/agentic_semantic_gap/semantic_agent_human_review.csv


,analysis_id,scope,batch_id,cluster_id,finding_id,classification,relation_family,cause_theme,effect_theme,gap_label,explanation,policy_evidence_ids,sentiment_evidence_ids,counterevidence_ids,evidence_quality,direct_causal_comparability,direct_comparability_share,policy_to_sentiment_deficit,sentiment_to_policy_deficit,classification_agreement,relation_agreement,representative_confidence,mean_confidence,mean_evidence_faithfulness,recurrent_runs,total_runs,usable_runs,run_recurrence,attempt_recurrence,finding_match_stability,evidence_jaccard_stability,member_finding_ids,machine_retained,human_is_gap,human_category,human_evidence_faithful,human_confirmed,human_notes
0,global__batch_01,global,1,global__batch_01__cluster_01,S1,partial_alignment,enables_or_supports,AI tools,teacher communication,AI support for teacher communication,Policy evidence states AI can assist in writin...,"[""clean_sentence_005462""]","[""clean_sentence_018986""]",[],adequate,True,1.0,0.400000,0.400000,1.0,1.000000,0.70,0.700,0.833333,3,4,3,1.000000,0.750000,0.627824,0.444444,"[""run_1:S1"", ""run_2:S1"", ""run_4:S1""]",True,,,,,
1,global__batch_01,global,1,global__batch_01__cluster_02,S2,policy_gap,risks_or_threatens,AI use,assessment integrity,Policy prohibits AI for assessments,Policy explicitly states students must not use...,"[""clean_sentence_014409""]","[""clean_sentence_019412""]",[],adequate,True,1.0,0.550000,0.350000,0.5,1.000000,0.60,0.650,0.850000,2,4,3,0.666667,0.500000,0.564344,0.333333,"[""run_1:S2"", ""run_2:S2""]",False,,,,,
2,global__batch_01,global,1,global__batch_01__cluster_03,S3,sentiment_gap,risks_or_threatens,AI application,education essence,Sentiment sees AI as threat to education,Sentiment evidence states AI is a threat to th...,"[""clean_sentence_000057""]","[""clean_sentence_017180""]",[],adequate,True,1.0,0.300000,0.700000,1.0,1.000000,0.60,0.600,0.800000,1,4,3,0.333333,0.250000,0.000000,0.000000,"[""run_1:S3""]",False,,,,,
3,global__batch_01,global,1,global__batch_01__cluster_04,S3,partial_alignment,requires_or_depends_on,teacher training,effective AI use,Teacher training for AI use,Policy evidence indicates teachers are expecte...,"[""clean_sentence_002786""]","[""clean_sentence_019187""]",[],adequate,True,1.0,0.350000,0.350000,1.0,1.000000,0.70,0.700,0.850000,2,4,3,0.666667,0.500000,0.875000,1.000000,"[""run_1:S4"", ""run_2:S3""]",True,,,,,
4,global__batch_01,global,1,global__batch_01__cluster_05,S4,partial_alignment,risks_or_threatens,AI over-reliance,student learning,Risk of pupil over-reliance on AI,Sentiment evidence explicitly lists risk of pu...,"[""clean_sentence_006935""]","[""clean_sentence_019412""]",[],adequate,True,1.0,0.500000,0.500000,1.0,1.000000,0.60,0.600,0.900000,1,4,3,0.333333,0.250000,0.000000,0.000000,"[""run_2:S4""]",False,,,,,
5,global__batch_01,global,1,global__batch_01__cluster_06,S5,partial_alignment,enables_or_supports,AI tools,student learning,AI support for student learning,Policy evidence shows AI can assist students i...,"[""clean_sentence_003187""]","[""clean_sentence_019002""]",[],adequate,True,1.0,0.400000,0.400000,1.0,1.000000,0.60,0.600,0.900000,1,4,3,0.333333,0.250000,0.000000,0.000000,"[""run_2:S5""]",False,,,,,
6,global__batch_02,global,2,global__batch_02__cluster_01,S1,partial_alignment,risks_or_threatens,AI-assisted learning,critical thinking,AI reduces critical thinking,Policy evidence states AI can enhance decision...,"[""clean_sentence_008623""]","[""clean_sentence_017817""]",[],adequate,True,1.0,0.666667,0.666667,1.0,0.666667,0.80,0.800,0.833333,3,3,3,1.000000,1.000000,0.877778,1.000000,"[""run_1:S1"", ""run_2:S1"", ""run_3:S1""]",True,,,,,
7,global__batch_02,global,2,global__batch_02__cluster_02,S2,partial_alignment,enables_or_supports,AI training and literacy,effective AI use,Training for effective AI use,Policy evidence mentions training and literacy...,"[""clean_sentence_000825""]","[""clean_sentence_019440""]",[],adequate,True,1.0,0.550000,0.400000,0.5,1.000000,0.70,0.700,0.850

,labelled_findings,true_positives,false_positives,false_negatives,precision,recall,f1,status
0,0,0,0,0,NaN,NaN,NaN,enter yes/no labels in human_is_gap


In [9]:
# Direction summaries, native-topic diagnostics, and report-ready PNG files.
# Blue/orange deliberately preserve the directional convention used elsewhere in the study.
BLUE = "#1f77b4"
ORANGE = "#ff7f0e"

if findings_df.empty:
    direction_summary = pd.DataFrame()
    remove_stale(DIRECTION_SUMMARY_PATH)
else:
    direction_map = {
        "policy_gap": "policy_to_sentiment",
        "sentiment_gap": "sentiment_to_policy",
        "partial_alignment": "bidirectional_partial_alignment",
        "alignment": "bidirectional_alignment",
    }
    table = findings_df.copy()
    table["direction"] = table["classification"].map(direction_map)
    direction_summary = table.groupby(["scope", "direction"], as_index=False).agg(
        findings=("cluster_id", "count"),
        mean_confidence=("mean_confidence", "mean"),
        mean_stability=("finding_match_stability", "mean"),
        mean_classification_agreement=("classification_agreement", "mean"),
    )
    direction_summary.to_csv(DIRECTION_SUMMARY_PATH, index=False)


def parse_json_list(value: Any) -> list[str]:
    if isinstance(value, list):
        return [str(item) for item in value]
    try:
        parsed = json.loads(str(value))
        return [str(item) for item in parsed] if isinstance(parsed, list) else []
    except Exception:
        return []


def evidence_topic_lookup() -> dict[str, dict[str, Any]]:
    columns = ["sentence_id", "corpus", "native_topic", "topic_code", "topic_space", "analysis_country"]
    frame = pd.concat([empirical[columns], synthetic[columns]], ignore_index=True)
    frame = frame.drop_duplicates("sentence_id")
    return frame.set_index("sentence_id").to_dict("index")


TOPIC_META = evidence_topic_lookup()


def build_native_topic_diagnostics(findings: pd.DataFrame):
    detail_rows=[]
    pair_rows=[]
    deficit_rows=[]
    if findings.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    for _, finding in findings.iterrows():
        policy_ids=parse_json_list(finding.get("policy_evidence_ids", "[]"))
        sentiment_ids=parse_json_list(finding.get("sentiment_evidence_ids", "[]"))
        p_score=pd.to_numeric(pd.Series([finding.get("policy_to_sentiment_deficit")]), errors="coerce").iloc[0]
        s_score=pd.to_numeric(pd.Series([finding.get("sentiment_to_policy_deficit")]), errors="coerce").iloc[0]

        policy_topics=sorted({
            str(TOPIC_META[eid]["topic_code"]) for eid in policy_ids
            if eid in TOPIC_META
            and pd.notna(TOPIC_META[eid].get("native_topic"))
            and str(TOPIC_META[eid].get("topic_code", "")).lower() not in {"", "nan", "none"}
        })
        sentiment_topics=sorted({
            str(TOPIC_META[eid]["topic_code"]) for eid in sentiment_ids
            if eid in TOPIC_META
            and pd.notna(TOPIC_META[eid].get("native_topic"))
            and str(TOPIC_META[eid].get("topic_code", "")).lower() not in {"", "nan", "none"}
        })

        for direction, ids, score in [
            ("policy_to_sentiment", policy_ids, p_score),
            ("sentiment_to_policy", sentiment_ids, s_score),
        ]:
            if not np.isfinite(score):
                continue
            deficit_rows.append({
                "analysis_id": finding["analysis_id"],
                "scope": finding["scope"],
                "cluster_id": finding["cluster_id"],
                "direction": direction,
                "agentic_semantic_deficit": float(score),
            })
            for evidence_id in ids:
                meta=TOPIC_META.get(str(evidence_id))
                if (
                    not meta
                    or pd.isna(meta.get("native_topic"))
                    or str(meta.get("topic_code", "")).lower() in {"", "nan", "none"}
                ):
                    continue
                detail_rows.append({
                    "analysis_id": finding["analysis_id"],
                    "scope": finding["scope"],
                    "cluster_id": finding["cluster_id"],
                    "direction": direction,
                    "evidence_id": evidence_id,
                    "topic_space": meta["topic_space"],
                    "native_topic": int(meta["native_topic"]),
                    "topic_code": meta["topic_code"],
                    "analysis_country": meta["analysis_country"],
                    "agentic_semantic_deficit": float(score),
                })

        # A pair is created only because the retained agentic finding itself links
        # evidence from these native policy and sentiment topics. No external topic
        # connection table is consumed.
        if np.isfinite(p_score) and np.isfinite(s_score):
            for p_code in policy_topics:
                for s_code in sentiment_topics:
                    pair_rows.append({
                        "analysis_id": finding["analysis_id"],
                        "scope": finding["scope"],
                        "cluster_id": finding["cluster_id"],
                        "policy_topic_code": p_code,
                        "sentiment_topic_code": s_code,
                        "pair_code": f"{p_code} ↔ {s_code}",
                        "policy_to_sentiment_deficit": float(p_score),
                        "sentiment_to_policy_deficit": float(s_score),
                    })

    detail=pd.DataFrame(detail_rows)
    pairs=pd.DataFrame(pair_rows)
    deficits=pd.DataFrame(deficit_rows)

    if detail.empty:
        return detail, pd.DataFrame(), pd.DataFrame(), deficits

    global_summary=detail.groupby(
        ["direction", "topic_space", "native_topic", "topic_code"], as_index=False
    ).agg(
        findings=("cluster_id", "nunique"),
        evidence_sentences=("evidence_id", "nunique"),
        mean_agentic_semantic_deficit=("agentic_semantic_deficit", "mean"),
    )

    country_detail=detail[detail["scope"].ne("global")].copy()
    country_summary=(
        country_detail.groupby(
            ["scope", "direction", "topic_space", "native_topic", "topic_code"], as_index=False
        ).agg(
            findings=("cluster_id", "nunique"),
            evidence_sentences=("evidence_id", "nunique"),
            mean_agentic_semantic_deficit=("agentic_semantic_deficit", "mean"),
        ) if not country_detail.empty else pd.DataFrame()
    )

    pair_summary=(
        pairs.groupby(["policy_topic_code", "sentiment_topic_code", "pair_code"], as_index=False).agg(
            findings=("cluster_id", "nunique"),
            policy_to_sentiment_deficit=("policy_to_sentiment_deficit", "mean"),
            sentiment_to_policy_deficit=("sentiment_to_policy_deficit", "mean"),
        ) if not pairs.empty else pd.DataFrame()
    )
    return global_summary, country_summary, pair_summary, deficits


native_topic_deficit_df, country_native_topic_deficit_df, native_topic_connections_df, directional_deficits_df = build_native_topic_diagnostics(findings_df)
write_nonempty_csv(native_topic_deficit_df, NATIVE_TOPIC_DEFICIT_PATH)
write_nonempty_csv(country_native_topic_deficit_df, COUNTRY_NATIVE_TOPIC_DEFICIT_PATH)
write_nonempty_csv(native_topic_connections_df, NATIVE_TOPIC_CONNECTION_PATH)
write_nonempty_csv(directional_deficits_df, DEFICIT_DISTRIBUTION_PATH)

generated_pngs=[]
skipped_pngs=[]


def save_figure(fig, path: Path) -> None:
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    generated_pngs.append(path)


def skip_figure(path: Path, reason: str) -> None:
    remove_stale(path)
    skipped_pngs.append((path, reason))


# PNG 1: empirical input support.
input_support_path=IMG_DIR/"agentic_semantic_input_support.png"
fig,ax=plt.subplots(figsize=(7.5,4.5))
plot_support=support.set_index("corpus").reindex(["policy", "sentiment"]).fillna(0)
ax.bar(plot_support.index, plot_support["sentences"])
ax.set_ylabel("Clean sentences")
ax.set_title("Agentic semantic input support")
save_figure(fig,input_support_path)

# PNG 2: causal-hint sampling diagnostic.
hint_path=IMG_DIR/"agentic_semantic_causal_hint_rate.png"
fig,ax=plt.subplots(figsize=(7.5,4.5))
ax.bar(plot_support.index, plot_support["causal_hint_rate"]*100)
ax.set_ylabel("Sentences with causal-language hint (%)")
ax.set_title("Causal-language sampling hints")
save_figure(fig,hint_path)

# PNG 3: validated candidate versus retained classifications.
classification_path=IMG_DIR/"agentic_semantic_classification_counts.png"
if candidates_df.empty:
    skip_figure(classification_path,"no validated empirical candidate clusters")
else:
    fig,ax=plt.subplots(figsize=(9.0,5.0))
    order=["policy_gap","sentiment_gap","partial_alignment","alignment"]
    candidate_counts=candidates_df["classification"].value_counts().reindex(order,fill_value=0)
    retained_counts=(
        findings_df["classification"].value_counts().reindex(order,fill_value=0)
        if not findings_df.empty else pd.Series(0,index=order)
    )
    x=np.arange(len(order)); width=0.36
    ax.bar(x-width/2,candidate_counts.values,width,label="Validated candidates")
    ax.bar(x+width/2,retained_counts.values,width,label="Retained recurrent")
    ax.set_xticks(x); ax.set_xticklabels(order,rotation=25,ha="right")
    ax.set_ylabel("Finding clusters")
    ax.set_title("Agentic semantic finding classifications")
    ax.legend()
    save_figure(fig,classification_path)

# PNG 4: confidence versus cross-run finding stability.
stability_path=IMG_DIR/"agentic_semantic_confidence_stability.png"
if findings_df.empty:
    skip_figure(stability_path,"no retained recurrent empirical findings")
else:
    fig,ax=plt.subplots(figsize=(7.5,4.8))
    ax.scatter(findings_df["finding_match_stability"],findings_df["mean_confidence"],s=45)
    ax.set_xlim(0,1.02); ax.set_ylim(0,1.02)
    ax.set_xlabel("Cross-run underlying-finding stability")
    ax.set_ylabel("Mean verifier confidence")
    ax.set_title("Agentic semantic confidence and stability")
    save_figure(fig,stability_path)

# PNG 5: synthetic perturbation robustness, explicitly including global and all
# eligible country scopes. Scope bars are means; points show the contributing batches.
robustness_path=IMG_DIR/"agentic_semantic_synthetic_robustness.png"
available=(
    robustness_summary_df[robustness_summary_df["status"].eq("available")].copy()
    if not robustness_summary_df.empty else pd.DataFrame()
)
if available.empty:
    skip_figure(robustness_path,"no usable baseline/probe robustness profiles")
else:
    available["profile_change"]=1.0-available["profile_jaccard"]
    preferred=["global","france","ireland"]
    scope_order=[scope for scope in preferred if scope in set(available["scope"])]
    scope_order += [scope for scope in sorted(set(available["scope"])) if scope not in scope_order]
    summary=available.groupby("scope",as_index=False).agg(
        classification_tvd=("classification_tvd","mean"),
        relation_tvd=("relation_tvd","mean"),
        profile_change=("profile_change","mean"),
    ).set_index("scope").reindex(scope_order)
    fig,ax=plt.subplots(figsize=(9.8,5.4))
    x=np.arange(len(summary)); width=0.24
    ax.bar(x-width,summary["classification_tvd"],width,label="Classification TVD")
    ax.bar(x,summary["relation_tvd"],width,label="Relation TVD")
    ax.bar(x+width,summary["profile_change"],width,label="Profile change (1-Jaccard)")
    for scope_index,scope in enumerate(scope_order):
        scope_rows=available[available["scope"].eq(scope)]
        for offset,column in [(-width,"classification_tvd"),(0.0,"relation_tvd"),(width,"profile_change")]:
            ax.scatter(np.full(len(scope_rows),scope_index+offset),scope_rows[column],s=20,zorder=3)
    ax.set_xticks(x); ax.set_xticklabels(scope_order)
    ax.set_ylim(0,1.02)
    ax.set_ylabel("Perturbation-induced output change")
    ax.set_title("Synthetic perturbation sensitivity by analysis scope")
    ax.legend()
    save_figure(fig,robustness_path)

# PNG 6: blue/orange native-topic connection bars. The visual layout mirrors the
# paired native-topic presentation used elsewhere, but the values are verifier-judged
# agentic deficits rather than embedding-derived coverage deficits.
native_topic_path=IMG_DIR/"global_agentic_semantic_deficit_by_topic_id.png"
if native_topic_connections_df.empty:
    skip_figure(native_topic_path,"no retained findings linking native policy and sentiment topics")
else:
    plot=native_topic_connections_df.sort_values(
        ["policy_topic_code","sentiment_topic_code"]
    ).reset_index(drop=True)
    x=np.arange(len(plot)); width=0.36
    fig,ax=plt.subplots(figsize=(max(12.0,0.9*len(plot)),7.0))
    ax.bar(x-width/2,plot["policy_to_sentiment_deficit"],width,color=BLUE,label="policy_to_sentiment")
    ax.bar(x+width/2,plot["sentiment_to_policy_deficit"],width,color=ORANGE,label="sentiment_to_policy")
    ax.set_xticks(x); ax.set_xticklabels(plot["pair_code"],rotation=90)
    ax.set_ylim(0,1.02)
    ax.set_xlabel("Agent-linked native policy ↔ sentiment topic IDs")
    ax.set_ylabel("Mean agent-judged semantic deficit")
    ax.set_title("Agentic semantic deficit by native topic connection")
    ax.legend(loc="upper right")
    save_figure(fig,native_topic_path)

# PNG 7: directional deficit histogram with the same blue/orange direction convention
# and a fixed 0--1 axis so visual comparisons remain straightforward.
deficit_hist_path=IMG_DIR/"agentic_semantic_deficit_histogram.png"
if directional_deficits_df.empty:
    skip_figure(deficit_hist_path,"no retained directional agentic deficit scores")
else:
    policy_vals=directional_deficits_df.loc[
        directional_deficits_df["direction"].eq("policy_to_sentiment"),"agentic_semantic_deficit"
    ].dropna().to_numpy(dtype=float)
    sentiment_vals=directional_deficits_df.loc[
        directional_deficits_df["direction"].eq("sentiment_to_policy"),"agentic_semantic_deficit"
    ].dropna().to_numpy(dtype=float)
    if len(policy_vals)==0 or len(sentiment_vals)==0:
        skip_figure(deficit_hist_path,"both directional deficit sets are required")
    else:
        bins=np.linspace(0.0,1.0,16)
        fig,ax=plt.subplots(figsize=(10.5,6.0))
        ax.hist(policy_vals,bins=bins,color=BLUE,alpha=0.52,label="policy_to_sentiment")
        ax.hist(sentiment_vals,bins=bins,color=ORANGE,alpha=0.52,label="sentiment_to_policy")
        ax.set_xlim(0,1.0)
        ax.set_xlabel("Agent-judged semantic deficit")
        ax.set_ylabel("Retained finding-direction scores")
        ax.legend(loc="upper right")
        ax.set_title("Agentic directional semantic deficit distribution")
        save_figure(fig,deficit_hist_path)

# PNG 8: country-level native-topic deficits. Each label is a country/topic code;
# colours preserve direction without implying that Pn and Sn with the same number align.
country_topic_path=IMG_DIR/"country_agentic_semantic_deficit_by_topic_id.png"
if country_native_topic_deficit_df.empty:
    skip_figure(country_topic_path,"no retained country findings with native-topic evidence")
else:
    plot=country_native_topic_deficit_df.sort_values(["scope","direction","native_topic"]).copy()
    labels=[]; blue_values=[]; orange_values=[]
    for _,row in plot.iterrows():
        labels.append(f"{row['scope']}:{row['topic_code']}")
        if row["direction"]=="policy_to_sentiment":
            blue_values.append(row["mean_agentic_semantic_deficit"]); orange_values.append(np.nan)
        else:
            blue_values.append(np.nan); orange_values.append(row["mean_agentic_semantic_deficit"])
    x=np.arange(len(labels)); b=np.asarray(blue_values,dtype=float); o=np.asarray(orange_values,dtype=float)
    fig,ax=plt.subplots(figsize=(max(12.0,0.72*len(labels)),6.5))
    bm=np.isfinite(b); om=np.isfinite(o)
    ax.bar(x[bm],b[bm],0.62,color=BLUE,label="policy_to_sentiment")
    ax.bar(x[om],o[om],0.62,color=ORANGE,label="sentiment_to_policy")
    ax.set_xticks(x); ax.set_xticklabels(labels,rotation=90)
    ax.set_ylim(0,1.02)
    ax.set_xlabel("Country and native frozen NMF topic ID")
    ax.set_ylabel("Mean agent-judged semantic deficit")
    ax.set_title("Country-level agentic semantic deficit by native topic ID")
    ax.legend(loc="upper right")
    save_figure(fig,country_topic_path)

# Validate every generated image by file size and decoder readability.
from PIL import Image
png_validation_rows=[]
for path in generated_pngs:
    readable=False; width=0; height=0; mode=""
    if path.exists() and path.stat().st_size>0:
        try:
            with Image.open(path) as image:
                image.verify()
            with Image.open(path) as image:
                image.load(); width,height=image.size; mode=image.mode; readable=True
        except Exception:
            readable=False
    png_validation_rows.append({
        "file":path.name,"generated":True,"bytes":path.stat().st_size if path.exists() else 0,
        "readable":readable,"width":width,"height":height,"mode":mode,
        "status":"passed" if readable else "failed","reason":"",
    })
for path,reason in skipped_pngs:
    png_validation_rows.append({
        "file":path.name,"generated":False,"bytes":0,"readable":False,
        "width":0,"height":0,"mode":"","status":"skipped","reason":reason,
    })
png_validation=pd.DataFrame(png_validation_rows)
png_validation.to_csv(PNG_VALIDATION_PATH,index=False)
if (png_validation["status"]=="failed").any():
    raise ValueError("One or more generated PNG diagnostics failed decoder validation.")

if not direction_summary.empty: display(direction_summary)
if not native_topic_connections_df.empty: display(native_topic_connections_df)
if not robustness_scope_summary_df.empty: display(robustness_scope_summary_df)
display(png_validation)


,scope,direction,findings,mean_confidence,mean_stability,mean_classification_agreement
0,france,bidirectional_partial_alignment,8,0.690625,0.714655,1.000000
1,global,bidirectional_alignment,1,0.850000,0.670690,1.000000
2,global,bidirectional_partial_alignment,3,0.733333,0.793534,1.000000
3,ireland,bidirectional_partial_alignment,2,0.783333,0.874464,0.833333
4,ireland,sentiment_to_policy,2,0.737500,0.641293,0.833333


,policy_topic_code,sentiment_topic_code,pair_code,findings,policy_to_sentiment_deficit,sentiment_to_policy_deficit
0,P2,S6,P2 ↔ S6,2,0.533333,0.533333
1,P2,S7,P2 ↔ S7,1,0.500000,0.500000
2,P2,S8,P2 ↔ S8,5,0.526667,0.486667
3,P3,S2,P3 ↔ S2,1,0.350000,0.350000
4,P3,S8,P3 ↔ S8,1,0.500000,0.500000
5,P5,S5,P5 ↔ S5,1,0.400000,0.400000
6,P6,S1,P6 ↔ S1,1,0.233333,0.233333
7,P7,S0,P7 ↔ S0,1,0.400000,0.400000
8,P7,S1,P7 ↔ S1,1,0.150000,0.150000
9,P7,S3,P7 ↔ S3,1,0.666667,0.666667


,scope,batches,mean_classification_tvd,mean_relation_tvd,mean_profile_change,mean_confidence_change,mean_faithfulness_change,mean_policy_to_sentiment_deficit_change,mean_sentiment_to_policy_deficit_change
0,france,3,0.133333,0.306566,0.540476,0.048586,-0.027778,-0.108889,-0.121212
1,global,3,0.261905,0.383045,0.634921,0.008034,-0.005556,-0.035563,-0.042085
2,ireland,3,0.420286,0.328620,0.692160,-0.024689,-0.005556,0.136515,-0.010699


,file,generated,bytes,readable,width,height,mode,status,reason
0,agentic_semantic_input_support.png,True,37902,True,1479,878,RGBA,passed,
1,agentic_semantic_causal_hint_rate.png,True,40100,True,1479,878,RGBA,passed,
2,agentic_semantic_classification_counts.png,True,69396,True,1779,977,RGBA,passed,
3,agentic_semantic_confidence_stability.png,True,52266,True,1480,938,RGBA,passed,
4,agentic_semantic_synthetic_robustness.png,True,71350,True,1940,1058,RGBA,passed,
5,global_agentic_semantic_deficit_by_topic_id.png,True,87510,True,2380,1380,RGBA,passed,
6,agentic_semantic_deficit_histogram.png,True,69255,True,2080,1178,RGBA,passed,
7,country_agentic_semantic_deficit_by_topic_id.png,True,97006,True,2380,1275,RGBA,passed,


In [10]:
result_paths = {
    "semantic_agent_evidence.csv": EVIDENCE_PATH,
    "semantic_agent_stability.csv": STABILITY_PATH,
    "semantic_agent_candidates.csv": CANDIDATES_PATH,
    "semantic_agent_findings.csv": FINDINGS_PATH,
    "semantic_agent_finding_evidence.csv": FINDING_EVIDENCE_PATH,
    "semantic_agent_human_review.csv": REVIEW_PATH,
    "semantic_agent_direction_summary.csv": DIRECTION_SUMMARY_PATH,
    "semantic_agent_robustness_summary.csv": ROBUSTNESS_SUMMARY_PATH,
    "semantic_agent_robustness_by_scope.csv": ROBUSTNESS_SCOPE_SUMMARY_PATH,
    "semantic_agent_deficit_by_native_topic.csv": NATIVE_TOPIC_DEFICIT_PATH,
    "semantic_agent_native_topic_connections.csv": NATIVE_TOPIC_CONNECTION_PATH,
    "semantic_agent_country_deficit_by_native_topic.csv": COUNTRY_NATIVE_TOPIC_DEFICIT_PATH,
    "semantic_agent_directional_deficits.csv": DEFICIT_DISTRIBUTION_PATH,
}

output_status_rows=[]
for name, path in result_paths.items():
    exists=path.exists()
    rows=None
    if exists and path.suffix == ".csv":
        try:
            rows=len(pd.read_csv(path))
        except Exception:
            rows=None
    output_status_rows.append({
        "file": name,
        "exists": exists,
        "rows": rows,
        "status": "available" if exists and (rows is None or rows > 0) else "not_generated",
    })

output_status=pd.DataFrame(output_status_rows)
output_status.to_csv(OUTPUT_STATUS_PATH, index=False)

run_summary=pd.DataFrame([{
    "method": "agentic_semantic_policy_sentiment_gap",
    "pipeline_version": PIPELINE_VERSION,
    "prompt_version": PROMPT_VERSION,
    "input_source": "output/shared/clean_sentence_inventory.csv",
    "cleaning_version": inventory["cleaning_version"].iloc[0],
    "shared_inventory_rows": len(inventory),
    "shared_inventory_sha256": metadata.get("inventory_sha256", ""),
    "empirical_rows": len(empirical),
    "policy_rows": int(empirical["corpus"].eq("policy").sum()),
    "sentiment_rows": int(empirical["corpus"].eq("sentiment").sum()),
    "synthetic_rows_for_robustness_only": len(synthetic),
    "analysis_packages": len(packages),
    "robustness_probe_packages": len(robustness_packages),
    "target_usable_runs_per_package": TARGET_USABLE_RUNS,
    "maximum_run_attempts_per_package": MAX_RUN_ATTEMPTS,
    "minimum_recurrent_runs": MIN_RECURRENT_RUNS,
    "minimum_classification_agreement": MIN_CLASSIFICATION_AGREEMENT,
    "minimum_relation_agreement": MIN_RELATION_AGREEMENT,
    "minimum_direct_comparability_share": MIN_DIRECT_COMPARABILITY_SHARE,
    "sentences_per_corpus_per_package": SENTENCES_PER_CORPUS,
    "causal_hint_share_target": CAUSAL_HINT_SHARE,
    "agent_enabled": RUN_AGENT,
    "usable_empirical_runs": sum(bool(row.get("run_usable")) for row in empirical_run_rows),
    "retained_recurrent_empirical_findings": len(findings_df),
    "synthetic_outputs_used_as_findings": False,
    "synthetic_robustness_scopes": ",".join(sorted({p["scope"] for p in robustness_packages})),
    "frozen_topics_source": "direct NMF assignments from the topic-modelling stage",
    "output_folder": str(OUTPUT_DIR),
}])
run_summary.to_csv(RUN_SUMMARY_PATH, index=False)

display(run_summary)
display(output_status)

print("Raw empirical agent runs:", RUNS_PATH)
print("Raw robustness agent runs:", ROBUSTNESS_RUNS_PATH if robustness_packages else "not generated")
print("PNG folder:", IMG_DIR)


,method,pipeline_version,prompt_version,input_source,cleaning_version,shared_inventory_rows,shared_inventory_sha256,empirical_rows,policy_rows,sentiment_rows,synthetic_rows_for_robustness_only,analysis_packages,robustness_probe_packages,target_usable_runs_per_package,maximum_run_attempts_per_package,minimum_recurrent_runs,minimum_classification_agreement,minimum_relation_agreement,minimum_direct_comparability_share,sentences_per_corpus_per_package,causal_hint_share_target,agent_enabled,usable_empirical_runs,retained_recurrent_empirical_findings,synthetic_outputs_used_as_findings,synthetic_robustness_scopes,frozen_topics_source,output_folder
0,agentic_semantic_policy_sentiment_gap,agentic-semantic-v3.1,direct-comparability-v3.1,output/shared/clean_sentence_inventory.csv,shared-causal-text,21591,59f25d414f8672b5fd0423a5c35ea1452e3d3d051a4d04...,19463,15281,4182,2128,9,9,3,5,2,0.66,0.5,0.66,24,0.85,True,27,16,False,"france,global,ireland",direct NMF assignments from the topic-modellin...,/home/nsirim/Github/mscdsa/msc/progress/causal...


,file,exists,rows,status
0,semantic_agent_evidence.csv,True,432,available
1,semantic_agent_stability.csv,True,9,available
2,semantic_agent_candidates.csv,True,58,available
3,semantic_agent_findings.csv,True,16,available
4,semantic_agent_finding_evidence.csv,True,34,available
5,semantic_agent_human_review.csv,True,58,available
6,semantic_agent_direction_summary.csv,True,5,available
7,semantic_agent_robustness_summary.csv,True,9,available
8,semantic_agent_robustness_by_scope.csv,True,3,available
9,semantic_agent_deficit_by_native_topic.csv,True,13,available


Raw empirical agent runs: /home/nsirim/Github/mscdsa/msc/progress/causal_nlp/output/agentic_semantic_gap/semantic_agent_runs.jsonl
Raw robustness agent runs: /home/nsirim/Github/mscdsa/msc/progress/causal_nlp/output/agentic_semantic_gap/semantic_agent_robustness_runs.jsonl
PNG folder: /home/nsirim/Github/mscdsa/msc/progress/causal_nlp/img/agentic_semantic_gap


## Interpretation

The empirical findings are retained only when the same underlying causal comparison recurs across usable analyst/verifier runs, passes evidence-faithfulness and confidence checks, and shows sufficient classification, relation, and direct-comparability agreement. The runner targets three usable repetitions and can make up to five attempts per batch, so an unusable service response does not automatically destroy a stability test. Synthetic sentiment is used only as a perturbation probe and is summarised globally and by every eligible country scope. Native-topic diagnostics use frozen NMF assignments loaded directly from the topic-modelling stage. Blue denotes policy-to-sentiment and orange denotes sentiment-to-policy; the plotted deficit is explicitly the verifier-judged agentic score.
